# Эксперимент 1: обычная LLaDA-8B-Instruct (HMM не используется)

Проект "Токенные зависимости в диффузионной генерации" (абляции метода CoDD, arXiv 2603.00045). Файл: `exp1_llada_default.ipynb`.
Запускать **только на GPU в Yandex DataSphere**. Режим: `use_pc=False` - слова-кандидаты и уверенность (значит, и порядок раскрытия) считаются по логитам нейросети, HMM не участвует.

> ## ЗАМЕТНЫЕ ПОМЕТКИ (прочитать до запуска)
> 1. **В блокнотах 2-5 (с HMM) заглушка pyjuice ЗАПРЕЩЕНА.** В exp1 путь HMM не вызывается, поэтому в аварийном случае (pyjuice не ставится) заглушка допустима (флаг `ALLOW_PYJUICE_STUB`, по умолчанию выключен), но тогда будет громкое предупреждение и запись `pyjuice: stub` в `env.json`.
> 2. **`HMM_CKPT` изменён по решению автора проекта**: в исходном задании было `il18/llada-math-pc`, в блокноте стоит `il18/llada-gsm-pc` (чекпоинт для GSM8K из таблицы README). В exp1 он не используется, константа общая для пяти блокнотов.
> 3. **`GEN_LENGTH = 512`** (единственная константа длины генерации): как в статье (приложение B) и в `eval/harness.py`; README-пример и `example.py` используют 256. Менять только по решению автора проекта. При 512 токенах прогон на T4 долгий (оценка времени печатается после дымового теста).
> 4. Блокнот **не запускался** при подготовке: проверен только синтаксис (`ast.parse`). Всё непроверенное помечено словами **НЕ ПРОВЕРЕНО**.
> 5. **T4 и dtype.** По умолчанию `DTYPE = "bfloat16"`, но T4 (compute capability 7.5) bf16 аппаратно не поддерживает. Блокнот выведет заметное предупреждение, но dtype сам **не меняет**: переход на `"float16"` - решение автора (численная устойчивость LLaDA в fp16 **НЕ ПРОВЕРЕНА**; для сопоставимости остальным четырём блокнотам нужен тот же dtype).

In [ ]:
# ===================== 0. КОНФИГ (первая ячейка с кодом) =====================
import os
import sys
import json
import hashlib

# ---------- ОБЩИЕ ПАРАМЕТРЫ (одинаковы у всех пяти блокнотов, менять нельзя) ----------
MODEL = "GSAI-ML/LLaDA-8B-Instruct"
HMM_CKPT = "il18/llada-gsm-pc"   # ИЗМЕНЕНО по решению автора проекта (в задании было "il18/llada-math-pc"). В exp1 НЕ используется.
DATASET = "gsm8k"
DATASET_SPLIT = "test"
N_TASKS = 100                    # первые 100 задач test-выборки в порядке датасета
GEN_LENGTH = 512                 # ЕДИНСТВЕННАЯ константа длины генерации (256 или 512). 512 = статья (приложение B) и eval/harness.py; README-пример и example.py: 256.
BLOCK_LENGTH = 32
TEMPERATURE = 0.0                # жадный выбор, как в статье
NUM_STEPS_LIST = [64, 128]
REMASKING_LIST = ["low_confidence", "margin"]
PC_TEMPERATURE = 0.2             # из CLI-примера README (example.py: 0.1; дефолт eval.py: 0.7). В exp1 не используется.
PC_FRAC = 0.5                    # из CLI-примера README (example.py: 0.7; дефолт eval.py: 0.3). В exp1 не используется.
REVERSE_FRAC = False             # дефолт eval.py. В exp1 не используется.
SEEDS = [0, 1, 2]                # в exp1 ИГНОРИРУЕТСЯ: случайности нет, один запуск на комбинацию (файлы с суффиксом _seed0)

# ---------- ПАРАМЕТРЫ ЭКСПЕРИМЕНТА 1 ----------
EXP_NAME = "exp1_llada_default"
USE_PC = False                   # обычная LLaDA: HMM не используется
SEED_USED = 0                    # значение поля seed в файлах результатов (случайности нет)
SYSTEM_INSTRUCTION = "You are a helpful assistant."   # eval.py: для gsm8k именно этот system prompt
NUM_FEWSHOT = 0                  # eval.py: --num_shot по умолчанию 0
TASK_NAME = "gsm8k"              # имя задачи в lm_eval (eval.py: TASKS['gsm8k'])
MASK_ID = 126336                 # CoDDConfig.mask_id
DTYPE = "bfloat16"               # "bfloat16" | "float16". eval.py: bfloat16. T4 (capability 7.5) bf16 аппаратно НЕ поддерживает: см. предупреждение в ячейке 1.1; dtype автоматически НЕ меняется.
ATTN_IMPLEMENTATION = "eager"    # eval.py: attn_implementation="eager"
N_BOOTSTRAP = 10000              # ресэмплов для доверительного интервала (по задачам)
BOOTSTRAP_SEED = 12345           # seed ТОЛЬКО для bootstrap (генерация случайности не использует)

# ---------- КОНФИГ СРЕДЫ DATASPHERE (путь, интернет и доступ к HF заранее неизвестны: НЕ ПРОВЕРЕНО) ----------
PROJECT_DIR_OVERRIDE = None      # явный путь к хранилищу проекта (высший приоритет); None = автоопределение
DATASPHERE_PROJECT_DIR = "/home/jupyter/datasphere/project"   # типичный путь проекта DataSphere (НЕ ПРОВЕРЕН)


def detect_project_dir():
    # Порядок: PROJECT_DIR_OVERRIDE (если задан) -> переменная окружения PROJECT_DIR -> стандартный путь DataSphere (если существует) -> текущая папка
    if PROJECT_DIR_OVERRIDE:
        return PROJECT_DIR_OVERRIDE, "PROJECT_DIR_OVERRIDE в конфиге"
    env_value = os.environ.get("PROJECT_DIR")
    if env_value:
        return env_value, "переменная окружения PROJECT_DIR"
    if os.path.isdir(DATASPHERE_PROJECT_DIR):
        return DATASPHERE_PROJECT_DIR, "стандартный путь проекта DataSphere (папка существует; что это именно хранилище проекта - НЕ ПРОВЕРЕНО)"
    return os.getcwd(), "текущая папка (запасной вариант: нет ни переменной окружения PROJECT_DIR, ни пути DataSphere)"


PROJECT_DIR, PROJECT_DIR_SOURCE = detect_project_dir()
print(f"PROJECT_DIR = {PROJECT_DIR}   <- {PROJECT_DIR_SOURCE}")
print("(путь не проверен: при необходимости задайте PROJECT_DIR_OVERRIDE или переменную окружения PROJECT_DIR)")
CODD_REPO_DIR = os.path.join(PROJECT_DIR, "CoDD")             # клон https://github.com/liuanji/CoDD (коммит d9760f4 или совместимый)
MODEL_PATH = os.path.join(PROJECT_DIR, "weights", "LLaDA-8B-Instruct")   # веса ТОЛЬКО здесь (одна копия); если их нет, их скачает ячейка 1.2a
HF_CACHE_DIR = os.path.join(PROJECT_DIR, ".hf_cache")         # HF_HOME и HF_HUB_CACHE направлены сюда (иначе кэш HF ляжет в домашнюю папку)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
if "huggingface_hub" in sys.modules or "transformers" in sys.modules:
    print("ВНИМАНИЕ: huggingface_hub/transformers уже импортированы в этом ядре - HF_HOME/HF_HUB_CACHE могут не примениться. Перезапустите ядро и выполните ячейки с начала.")
DATA_DIR = os.path.join(PROJECT_DIR, "data", "gsm8k")        # локальные файлы GSM8K, если Hugging Face недоступен
LOCAL_FILES_ONLY = None          # None = авто: True, если MODEL_PATH - локальная папка
HF_OFFLINE = False               # True: выставить HF_HUB_OFFLINE=1 (если Hugging Face недоступен или подвисает)
INSTALL_DEPS = True              # ставить недостающие пакеты через pip (torch не трогается)
PIP_FIND_LINKS = None            # путь к папке с колёсами (wheels) в хранилище проекта; тогда pip работает с --no-index
ALLOW_PYJUICE_STUB = False       # ТОЛЬКО exp1 и только аварийно. В блокнотах 2-5 заглушка ЗАПРЕЩЕНА.
ALLOW_MODEL_CODE_MISMATCH = False  # True: продолжить, даже если modeling_llada.py не совпал по sha256 с файлом из снапшота весов
STRICT_VERSIONS = False          # True: расхождение версий с CoDD/environment.yml - ошибка (по умолчанию предупреждение)
MIN_GPU_TOTAL_GB = 20            # минимум памяти GPU, ГиБ (gt4i.1: "24 ГБ" на деле ~23 доступно; 8B в 16-битном dtype ~ 15-16 ГиБ весов + активации)
MIN_GPU_FREE_GB = 20
RUN_LIMIT = None                 # None = все N_TASKS; число = обрабатывать только первые RUN_LIMIT задач в ПОЛНОМ прогоне (tasks.json остаётся из 100 задач)
MAX_RUNTIME_MIN = None           # None = без ограничения. Число минут: полный прогон после ближайшей завершённой задачи корректно остановится (кэш и steplog дописаны)
MIN_FREE_DISK_GB = 25            # перед скачиванием весов: свободно на диске не меньше (ГиБ). Лимит хранилища проекта 50 ГБ, включая пакеты
WEIGHTS_CACHE_WARN_GB = 20       # предупреждать, если weights + .hf_cache вместе больше (ГиБ): значит, веса продублированы
OUT_DIR = os.path.join(PROJECT_DIR, "results", EXP_NAME)    # results/exp1_llada_default/ в хранилище проекта

# ---------- ПРОВЕРКИ ПАРАМЕТРОВ ----------
assert MODEL == "GSAI-ML/LLaDA-8B-Instruct", "MODEL - общий параметр, менять нельзя"
assert HMM_CKPT == "il18/llada-gsm-pc", "HMM_CKPT - общий параметр (см. пометку 2 в начале блокнота)"
assert (DATASET, DATASET_SPLIT, N_TASKS) == ("gsm8k", "test", 100), "DATASET/DATASET_SPLIT/N_TASKS - общие параметры"
assert GEN_LENGTH in (256, 512), "GEN_LENGTH должен быть 256 или 512 (выбор автора проекта)"
assert BLOCK_LENGTH == 32 and TEMPERATURE == 0.0, "BLOCK_LENGTH/TEMPERATURE - общие параметры"
assert NUM_STEPS_LIST == [64, 128] and REMASKING_LIST == ["low_confidence", "margin"], "NUM_STEPS_LIST/REMASKING_LIST - общие параметры"
assert (PC_TEMPERATURE, PC_FRAC, REVERSE_FRAC) == (0.2, 0.5, False), "PC_* - общие параметры"
assert SEEDS == [0, 1, 2], "SEEDS - общий параметр (в exp1 игнорируется)"
assert USE_PC is False, "exp1 - это эксперимент БЕЗ HMM"
assert RUN_LIMIT is None or (isinstance(RUN_LIMIT, int) and 1 <= RUN_LIMIT <= N_TASKS), "RUN_LIMIT: None или целое от 1 до N_TASKS"
assert DTYPE in ("bfloat16", "float16"), f"DTYPE должен быть 'bfloat16' или 'float16', получено {DTYPE!r}"
assert MAX_RUNTIME_MIN is None or (isinstance(MAX_RUNTIME_MIN, (int, float)) and MAX_RUNTIME_MIN > 0), "MAX_RUNTIME_MIN: None или число минут > 0"

# Кратность: проверяем ОБА допустимых значения GEN_LENGTH, чтобы выбор автора не сломал запуск
for _gl in sorted({256, 512, GEN_LENGTH}):
    assert _gl % BLOCK_LENGTH == 0, f"GEN_LENGTH={_gl} не делится на BLOCK_LENGTH={BLOCK_LENGTH}"
    _nb = _gl // BLOCK_LENGTH
    for _ns in NUM_STEPS_LIST:
        assert _ns % _nb == 0, f"num_steps={_ns} не делится на число блоков {_nb} (GEN_LENGTH={_gl})"
NUM_BLOCKS = GEN_LENGTH // BLOCK_LENGTH
print(f"GEN_LENGTH={GEN_LENGTH}: блоков {NUM_BLOCKS}; токенов за шаг: "
      + ", ".join(f"{ns} шагов -> {BLOCK_LENGTH * NUM_BLOCKS / ns:g}" for ns in NUM_STEPS_LIST))


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")).hexdigest()


COMMON_PARAMS = {
    "MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET, "DATASET_SPLIT": DATASET_SPLIT, "N_TASKS": N_TASKS,
    "GEN_LENGTH": GEN_LENGTH, "BLOCK_LENGTH": BLOCK_LENGTH, "TEMPERATURE": TEMPERATURE,
    "NUM_STEPS_LIST": NUM_STEPS_LIST, "REMASKING_LIST": REMASKING_LIST,
    "PC_TEMPERATURE": PC_TEMPERATURE, "PC_FRAC": PC_FRAC, "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS,
}
EXP_PARAMS = {
    **COMMON_PARAMS,
    "EXP_NAME": EXP_NAME, "USE_PC": USE_PC, "SEED_USED": SEED_USED, "SYSTEM_INSTRUCTION": SYSTEM_INSTRUCTION,
    "NUM_FEWSHOT": NUM_FEWSHOT, "TASK_NAME": TASK_NAME, "MASK_ID": MASK_ID,
    "DTYPE": DTYPE, "ATTN_IMPLEMENTATION": ATTN_IMPLEMENTATION,
}
COMMON_SHA256 = sha256_json(COMMON_PARAMS)   # одинаков у всех блокнотов при одинаковых общих параметрах
EXP_SHA256 = sha256_json(EXP_PARAMS)
print("sha256 общих параметров:", COMMON_SHA256)
print("sha256 параметров эксперимента:", EXP_SHA256)

# ---------- ПУТИ ВЕСОВ ----------
MODEL_PATH_RESOLVED = MODEL_PATH     # всегда локальная папка в хранилище проекта; если весов в ней нет, их скачает ячейка 1.2a
LOCAL_FILES_ONLY_RESOLVED = None     # определяется в ячейке 1.2a после проверки/скачивания весов
if HF_OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
print("CODD_REPO_DIR:", CODD_REPO_DIR)
print("MODEL_PATH (веса, одна копия):", MODEL_PATH_RESOLVED)
print("HF_HOME:", os.environ["HF_HOME"], "| HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])
print("OUT_DIR:", OUT_DIR)
print("DTYPE:", DTYPE, "| GEN_LENGTH:", GEN_LENGTH, "| RUN_LIMIT:", RUN_LIMIT, "| MAX_RUNTIME_MIN:", MAX_RUNTIME_MIN, "| PIP_FIND_LINKS:", PIP_FIND_LINKS)
print()
print("ПОМЕТКИ: HMM_CKPT изменён на il18/llada-gsm-pc (по решению автора); GEN_LENGTH = 512 (как в статье);")
print("         T4 не поддерживает bf16 аппаратно: dtype не меняется молча, см. предупреждение в ячейке 1.1;")
print("         в блокнотах 2-5 заглушка pyjuice ЗАПРЕЩЕНА; блокнот не запускался при подготовке (только ast.parse).")

## 0. Заметка о коде (читал локальный клон CoDD, коммит d9760f4)

**Подтвердилось**
- `CoddLlada.forward` (`eval/llada/codd_llada.py`): вызывает базовую LLaDA, затем `apply_pc_logits(...)`, возвращает `CoDDOutput(logits=...)`. PC применяется только при `use_pc and pc_model is not None and pc_block_range is not None`; при `use_pc=False` логиты базовой модели не меняются.
- `apply_pc_logits` (`codd/model_class.py`): меняет логиты по месту только в блоке `[start_idx, end_idx)` и только если доля масок в блоке `< pc_frac` (или `>` при `reverse_frac=True`); делит логиты на `pc_temperature` (если `0 < t < 1`), делает прямой проход PC, `juice.queries.sample(conditional=True)`, затем добавляет log-параметры входных узлов.
- `llada_diffusion_generate` (`eval/llada/llada_generate.py`): на шаге forward -> logits -> `x0 = argmax(logits + gumbel)` -> уверенность `x0_p` по тем же логитам (low_confidence, random, margin, entropy) -> top-k по уверенности -> запись слов. При `temperature=0` шум не добавляется. Условия: `gen_length % block_length == 0`, `num_steps % num_blocks == 0`.
- README: флаги `--alg --num_steps --pc_ckpt --pc_temperature --pc_frac`; чекпоинты `il18/llada-math-pc`, `il18/llada-gsm-pc`, `il18/llada-code-pc`; оценка адаптирована из APD.
- Проверка ответов GSM8K делается кодом репозитория: `eval/eval.py` запускает задачу `gsm8k` из `lm-evaluation-harness` (0-shot, chat template, system "You are a helpful assistant."), фильтры `strict-match` и `flexible-extract`, метрика exact_match; `eval/utils.py::parse_results` берёт `flexible-extract`. Эти же задача, шаблон, фильтры и метрика используются ниже (свой код проверки не написан).

**Не подтвердилось / расхождения** (значения констант не менялись молча)
- В задании `HMM_CKPT = il18/llada-math-pc`, но датасет GSM8K; в README есть `il18/llada-gsm-pc`. Поставлен `il18/llada-gsm-pc` по решению автора. В exp1 не используется.
- Длина генерации: `eval/harness.py` жёстко ставит `gen_length=512`, статья (приложение B) - 512 токенов для всех задач; README-пример и `example.py` - 256. Здесь `GEN_LENGTH = 512` (как в статье и харнессе), это единственная константа.
- Значения `pc_temperature/pc_frac`: README-пример CLI 0.2/0.5, `example.py` 0.1/0.7, дефолты `eval.py` 0.7/0.3, дефолты `CoDDConfig` 0.1/0.7. На exp1 не влияет.
- Оригинальный харнесс без HMM (`LladaEvalHarness`) допускает только `low_confidence` и `random` (`assert` в `eval/harness.py`); `margin` без HMM он бы не запустил. Здесь функция генерации вызывается напрямую, ограничения нет.
- `_is_codd_model` по docstring проверяет, что PC загружен, а по коду - только наличие атрибутов. На exp1 не влияет.
- В `eval.py` обычная LLaDA грузится через `AutoModel(trust_remote_code)`, а с HMM - через `CoddLlada`. Здесь для всех режимов используется `CoddLlada` (единый код модели); совпадение исходников с кодом из снапшота весов проверяется по sha256 в разделе 2.
- В GSM8K нет поля `id`: `task_id` = индекс задачи в test-выборке (0..99).

**Версии** (`CoDD/environment.yml`, Python 3.12.12): torch 2.9.1, transformers 4.57.3, pyjuice 2.4.1, huggingface-hub 0.36.0, numpy 2.3.5, peft 0.18.1, tqdm 4.67.1, triton 3.5.1. `lm_eval` берётся из папки репозитория (`lm-evaluation-harness`, 0.4.10.dev0); версии `datasets`, `evaluate`, `accelerate` в `environment.yml` не заданы.

**НЕ ПРОВЕРЕНО** (причина: блокнот не запускался, нет доступа к DataSphere и GPU)
- совместимость torch/transformers/pyjuice с драйвером и CUDA в DataSphere, установка pyjuice там;
- доступность Hugging Face, pip и датасета GSM8K из DataSphere; путь хранилища проекта;
- работа `HFLM(pretrained=<CoddLlada>)` и всей цепочки подготовки промпта на реальной модели;
- совпадение `modeling_llada.py` из репозитория с файлом из снапшота весов (определяется в разделе 2; если файла рядом с весами нет - будет написано "НЕ ПРОВЕРЕНО");
- на каком коде посчитаны базовые цифры для `margin` в статье: оригинальный харнесс без HMM поддерживает только `low_confidence` и `random`;
- побитовая воспроизводимость на другом GPU; время полного прогона (оценка печатается после дымового теста);
- T4 (compute capability 7.5) и dtype: bf16 аппаратно не поддерживается (эмуляция или ошибка), численная устойчивость LLaDA в fp16 не проверена; dtype выбирает автор (`DTYPE`);
- путь хранилища проекта `/home/jupyter/datasphere/project` (взят по памяти, автоопределение с переопределением); соответствие `shutil.disk_usage` квоте проекта 50 ГБ;
- что `snapshot_download(local_dir=...)` в huggingface_hub 0.36 не оставляет вторую копию весов в кэше (поэтому после загрузки печатаются размеры `weights` и `.hf_cache`);
- совместимость установленных python/torch/transformers с remote-кодом модели и кодом CoDD (проверена только комбинация из `environment.yml`);
- ограничения сессии DataSphere по времени и порядок остановки ВМ в интерфейсе.

## 1. Проверка окружения и GPU, установка

Ячейки ниже: (1) проверяют Python, GPU, запись в хранилище, наличие репозитория; (2) ставят недостающие пакеты (если `INSTALL_DEPS`; torch и CUDA-стек не переустанавливаются); (2а) проверяют или скачивают веса в `MODEL_PATH`, направляют кэш HF в `PROJECT_DIR/.hf_cache` и считают занятое место (лимит хранилища проекта 50 ГБ); (3) импортируют pyjuice (или, аварийно, заглушку - только в exp1); (4) импортируют код репозитория и пишут `env.json` и `config.json`.

### Если Hugging Face или pip в DataSphere недоступны
1. **Веса** (~16 ГБ). Если Hugging Face доступен, ячейка 1.2a скачает их сама в `MODEL_PATH` (кэш HF направлен в `PROJECT_DIR/.hf_cache`, второй копии быть не должно). Иначе, на машине с интернетом: `huggingface-cli download GSAI-ML/LLaDA-8B-Instruct --local-dir LLaDA-8B-Instruct`. Загрузите папку в хранилище проекта, например `<PROJECT_DIR>/weights/LLaDA-8B-Instruct` (или укажите путь в `MODEL_PATH`). Внутри должны быть `config.json`, файлы весов (`*.safetensors`), файлы токенизатора и `modeling_llada.py`, `configuration_llada.py` (последние нужны для сверки кода в разделе 2).
2. **Репозиторий CoDD**: `git clone https://github.com/liuanji/CoDD` на машине с интернетом (или скачайте архив), положите папку в хранилище как `<PROJECT_DIR>/CoDD` (путь - `CODD_REPO_DIR`). Нужна вложенная папка `lm-evaluation-harness`. Желателен коммит `d9760f4`.
3. **Датасет GSM8K**: положите в `DATA_DIR` файлы `test` и `train` либо в виде `main/test-00000-of-00001.parquet` и `main/train-00000-of-00001.parquet` (структура репозитория `openai/gsm8k` на Hugging Face), либо `test.jsonl` и `train.jsonl` с полями `question` и `answer`. Если `train` нет, блокнот подставит `test` вместо него (на 0-shot это не влияет, будет предупреждение).
4. **pip недоступен**: на машине с интернетом, с тем же Python и платформой, скачайте колёса: `pip download -d wheels <список пакетов>` (список недостающих пакетов печатает ячейка установки), положите папку `wheels` в хранилище и задайте `PIP_FIND_LINKS = "<путь>/wheels"`. torch этим блокнотом не переустанавливается: используется образ DataSphere с PyTorch.
5. **pyjuice не ставится**: в блокнотах 2-5 это остановка (заглушка запрещена). Только в exp1 можно поставить `ALLOW_PYJUICE_STUB = True`.
6. Если нужной версии библиотеки нет - остановитесь и сообщите автору проекта; версии молча не менять.
7. Путь хранилища проекта DataSphere, наличие интернета и доступ к Hugging Face - **НЕ ПРОВЕРЕНО** (заранее неизвестны); ячейка проверки печатает фактическое состояние.

In [ ]:
# ===================== 1.1 ПРОВЕРКА ОКРУЖЕНИЯ И GPU =====================
import sys
import platform
import shutil
import time
import datetime
import subprocess
import importlib
import importlib.util
import importlib.metadata as importlib_metadata
import urllib.request


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def dist_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def probe_url(url, timeout=6):
    try:
        urllib.request.urlopen(url, timeout=timeout).close()
        return True
    except Exception as e:   # сеть может быть закрыта: это не ошибка блокнота, а факт о среде
        return f"нет ({type(e).__name__}: {e})"


START_UTC = utc_now()
print("Время старта (UTC):", START_UTC)
print("Python:", sys.version.split()[0], "|", platform.platform())
assert sys.version_info >= (3, 10), f"Нужен Python >= 3.10 (lm_eval из репозитория), у вас {sys.version.split()[0]}"

try:
    import torch
except Exception as e:
    raise RuntimeError("torch не импортируется. В DataSphere выберите образ с PyTorch и GPU; этим блокнотом torch не переустанавливается.") from e
print("torch:", torch.__version__, "| CUDA (сборка torch):", torch.version.cuda)
TORCH_DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE]
print("DTYPE (из конфига):", DTYPE)

assert torch.cuda.is_available(), (
    "GPU не найден (torch.cuda.is_available() == False). Блокнот работает только на GPU: "
    "выберите в DataSphere конфигурацию с GPU и перезапустите ядро.")
GPU_INDEX = torch.cuda.current_device()
_props = torch.cuda.get_device_properties(GPU_INDEX)
_free, _total = torch.cuda.mem_get_info()
GPU_NAME = _props.name
GPU_TOTAL_GB = _total / 1024 ** 3
GPU_FREE_GB = _free / 1024 ** 3
GPU_CAPABILITY = f"{_props.major}.{_props.minor}"
print(f"GPU: {GPU_NAME} | compute capability {GPU_CAPABILITY}")
print(f"Видеопамять фактически: всего {GPU_TOTAL_GB:.2f} ГиБ ({_total} байт), свободно сейчас {GPU_FREE_GB:.2f} ГиБ ({_free} байт)")
assert GPU_TOTAL_GB >= MIN_GPU_TOTAL_GB, f"Памяти GPU {GPU_TOTAL_GB:.2f} ГиБ < MIN_GPU_TOTAL_GB={MIN_GPU_TOTAL_GB}: 8B-модель в 16-битном dtype не поместится с запасом."
assert GPU_FREE_GB >= MIN_GPU_FREE_GB, f"Свободно {GPU_FREE_GB:.2f} ГиБ < MIN_GPU_FREE_GB={MIN_GPU_FREE_GB}: освободите GPU (другие ядра/процессы)."
if _props.major < 8:
    print("!" * 100)
    print(f"ЗАМЕТНОЕ ПРЕДУПРЕЖДЕНИЕ: у GPU compute capability {GPU_CAPABILITY} < 8.0 (T4 = 7.5): bfloat16 АППАРАТНО НЕ ПОДДЕРЖИВАЕТСЯ.")
    print(f"   Выбран DTYPE = {DTYPE!r}. dtype автоматически НЕ меняется.")
    if DTYPE == "bfloat16":
        print("   bfloat16 на этом GPU может эмулироваться (очень медленно) или привести к ошибке (НЕ ПРОВЕРЕНО).")
        print("   Если нужен float16, поставьте DTYPE = 'float16' в конфиге сами - это решение автора проекта.")
    else:
        print("   float16 T4 поддерживает, но численная устойчивость LLaDA в fp16 НЕ ПРОВЕРЕНА (возможны NaN/inf; дымовой тест проверит конечность логитов).")
        print("   Результаты fp16 и bf16 могут отличаться; остальным четырём блокнотам для сопоставимости нужен тот же dtype.")
    print("!" * 100)

# --- хранилище проекта ---
assert os.path.isdir(PROJECT_DIR), f"PROJECT_DIR не существует: {PROJECT_DIR}. Укажите путь хранилища проекта в конфиге."
os.makedirs(OUT_DIR, exist_ok=True)
_probe = os.path.join(OUT_DIR, ".write_probe")
with open(_probe, "w", encoding="utf-8") as _f:
    _f.write("ok")
os.remove(_probe)
print(f"Запись в OUT_DIR работает: {OUT_DIR} | свободно на диске {shutil.disk_usage(OUT_DIR).free / 1024 ** 3:.1f} ГБ")

# --- репозиторий CoDD ---
REQUIRED_REPO_FILES = [
    "eval/llada/codd_llada.py", "eval/llada/llada_generate.py", "eval/llada/modeling_llada.py",
    "eval/llada/configuration_llada.py", "codd/model_class.py", "codd/__init__.py",
    "eval/eval.py", "eval/harness.py", "lm-evaluation-harness/lm_eval/__init__.py",
    "lm-evaluation-harness/lm_eval/tasks/gsm8k/gsm8k.yaml", "lm-evaluation-harness/pyproject.toml",
]
_missing_files = [p for p in REQUIRED_REPO_FILES if not os.path.isfile(os.path.join(CODD_REPO_DIR, p))]
assert not _missing_files, (
    f"В CODD_REPO_DIR={CODD_REPO_DIR} нет файлов: {_missing_files}. "
    "Положите клон репозитория CoDD (с папкой lm-evaluation-harness) в хранилище проекта - см. инструкцию выше.")
print("Репозиторий CoDD найден:", CODD_REPO_DIR)

# --- сеть (только печать фактов) ---
HF_REACHABLE = probe_url("https://huggingface.co")
PIP_REACHABLE = probe_url("https://pypi.org/simple/pip/")
print("Hugging Face доступен:", HF_REACHABLE)
print("PyPI доступен:", PIP_REACHABLE)

# --- веса: здесь только факт; проверку полноты и скачивание делает ячейка 1.2a ---
if os.path.isdir(MODEL_PATH_RESOLVED) and os.listdir(MODEL_PATH_RESOLVED):
    print("В MODEL_PATH уже есть файлы:", MODEL_PATH_RESOLVED, f"({len(os.listdir(MODEL_PATH_RESOLVED))} шт.); полноту проверит ячейка 1.2a")
else:
    print("В MODEL_PATH весов пока нет:", MODEL_PATH_RESOLVED, "- ячейка 1.2a скачает их (нужен Hugging Face) либо положите вручную (инструкция выше).")

ENV_INFO = {
    "started_utc": START_UTC,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "gpu_name": GPU_NAME,
    "gpu_total_gb": round(GPU_TOTAL_GB, 2),
    "gpu_capability": GPU_CAPABILITY,
    "hf_reachable": HF_REACHABLE,
    "pypi_reachable": PIP_REACHABLE,
    "gpu_total_bytes": _total,
    "dtype": DTYPE,
    "project_dir": PROJECT_DIR,
    "project_dir_source": PROJECT_DIR_SOURCE,
    "model_path": MODEL_PATH_RESOLVED,
    "hf_cache_dir": HF_CACHE_DIR,
}
print("OK: окружение и GPU проверены.")

In [ ]:
# ===================== 1.2 УСТАНОВКА НЕДОСТАЮЩИХ ПАКЕТОВ И СВЕРКА ВЕРСИЙ =====================
# Версии взяты из CoDD/environment.yml, где они заданы; иначе - минимум из pyproject lm_eval.
PIP_SPECS = {
    # имя для import : требование pip
    "transformers": "transformers==4.57.3",
    "huggingface_hub": "huggingface-hub==0.36.0",
    "accelerate": "accelerate>=0.26.0",
    "peft": "peft==0.18.1",
    "tqdm": "tqdm==4.67.1",
    "numpy": "numpy==2.3.5",
    "datasets": "datasets>=2.16.0",
    "evaluate": "evaluate>=0.4.0",
    "jinja2": "jinja2",
    "jsonlines": "jsonlines",
    "pytablewriter": "pytablewriter",
    "rouge_score": "rouge-score>=0.0.4",
    "sacrebleu": "sacrebleu>=1.5.0",
    "sklearn": "scikit-learn>=0.24.1",
    "sqlitedict": "sqlitedict",
    "zstandard": "zstandard",
    "dill": "dill",
    "word2number": "word2number",
    "more_itertools": "more_itertools",
}
EXPECTED_VERSIONS = {   # distribution name -> версия из environment.yml
    "torch": "2.9.1", "transformers": "4.57.3", "pyjuice": "2.4.1", "huggingface-hub": "0.36.0",
    "numpy": "2.3.5", "peft": "0.18.1", "tqdm": "4.67.1", "triton": "3.5.1",
}


def missing_specs():
    return [spec for mod, spec in PIP_SPECS.items() if importlib.util.find_spec(mod) is None]


def write_pip_constraints():
    # Файл constraints: pip не имеет права менять уже установленные torch, CUDA-стек (nvidia-*), triton, numpy, transformers и др.
    # Если зависимость требует другой версии, pip завершится ошибкой, а не переустановит пакет.
    keep = ["torch", "torchvision", "torchaudio", "triton", "numpy", "transformers", "tokenizers", "huggingface-hub", "safetensors"]
    pins = [f"{n}=={dist_version(n)}" for n in keep if dist_version(n)]
    for dist in importlib_metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().startswith("nvidia-"):
            pins.append(f"{name}=={dist.version}")
    path = os.path.join(OUT_DIR, "pip_constraints.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(sorted(set(pins))) + "\n")
    return path, pins


_missing = missing_specs()
if _missing:
    print("Не хватает пакетов (для pip download / PIP_FIND_LINKS):", _missing)
    assert INSTALL_DEPS, f"Не хватает пакетов {_missing}, а INSTALL_DEPS=False. Установите их вручную или включите INSTALL_DEPS."
    _cpath, _pins = write_pip_constraints()
    print(f"Зафиксировано (constraints) {len(_pins)} установленных пакетов, в т.ч. torch и CUDA-стек: они переустанавливаться не будут.")
    _cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-c", _cpath]   # --no-cache-dir: не раздувать хранилище проекта кэшем pip
    if PIP_FIND_LINKS:
        _cmd += ["--no-index", "--find-links", PIP_FIND_LINKS]
    _cmd += _missing
    print("Запуск:", " ".join(_cmd))
    _res = subprocess.run(_cmd, capture_output=True, text=True)
    print(_res.stdout[-2500:])
    if _res.returncode != 0:
        print(_res.stderr[-2500:])
        raise RuntimeError(
            "pip install завершился с ошибкой (см. вывод выше). Если pip/PyPI недоступны - используйте PIP_FIND_LINKS "
            "(инструкция в начале раздела 1). Версии молча не менять; сообщите автору проекта.")
    importlib.invalidate_caches()
    _still = missing_specs()
    assert not _still, f"После установки всё ещё не импортируются: {_still}"
else:
    print("Все нужные пакеты уже есть.")

# --- сверка версий с environment.yml ---
_py_have = sys.version.split()[0]
print(f"Версии среды: python {_py_have} | torch {torch.__version__} | transformers {dist_version('transformers')}")
print(f"{'пакет':<18}{'ожидается':<12}{'установлено':<14}статус")
_mismatch = []
_py_ok = _py_have.startswith("3.12")
if not _py_ok:
    _mismatch.append(("python", "3.12.x", _py_have))
print(f"{'python':<18}{'3.12.x':<12}{_py_have:<14}{'OK' if _py_ok else 'РАСХОЖДЕНИЕ'}")
for _name, _want in EXPECTED_VERSIONS.items():
    _have = dist_version(_name)
    _ok = (_have == _want)
    if not _ok and _name != "pyjuice":   # pyjuice проверяется в следующей ячейке
        _mismatch.append((_name, _want, _have))
    print(f"{_name:<18}{_want:<12}{str(_have):<14}{'OK' if _ok else 'РАСХОЖДЕНИЕ'}")
if _mismatch:
    _msg = f"Расхождения версий с CoDD/environment.yml (python 3.12, torch 2.9.1, transformers 4.57.3 и др.): {_mismatch}."
    if STRICT_VERSIONS:
        raise RuntimeError(_msg)
    print("!" * 100)
    print("ПРЕДУПРЕЖДЕНИЕ (не ошибка, работа продолжается):", _msg)
    print("!" * 100)
print("НЕ ПРОВЕРЕНО: совместимость установленных python/torch/transformers с remote-кодом модели (modeling_llada.py), "
      "кодом CoDD и lm_eval из репозитория; проверена только комбинация из environment.yml репозитория.")
ENV_INFO["version_mismatches_vs_environment_yml"] = [list(m) for m in _mismatch]

In [ ]:
# ===================== 1.2a ВЕСА В ХРАНИЛИЩЕ ПРОЕКТА, КЭШ HF, ДИСК =====================
# Веса лежат ТОЛЬКО в MODEL_PATH (PROJECT_DIR/weights/LLaDA-8B-Instruct). Кэш HF направлен в PROJECT_DIR/.hf_cache (см. конфиг).
GIB = 1024 ** 3


def dir_size_bytes(path):
    # размер папки без следования по симлинкам (lstat); ошибки доступа пропускаются
    total = 0
    for root, _dirs, files in os.walk(path):
        for name in files:
            try:
                total += os.lstat(os.path.join(root, name)).st_size
            except OSError:
                pass
    return total


def weights_present(path):
    if not os.path.isdir(path):
        return False
    names = os.listdir(path)
    return "config.json" in names and any(n.endswith((".safetensors", ".bin")) for n in names)


def fetch_repo_index(repo_id):
    # {имя файла: размер} репозитория на Hugging Face; None, если HF недоступен или API ответил ошибкой
    if HF_REACHABLE is not True:
        return None
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(repo_id, files_metadata=True)
        return {s.rfilename: s.size for s in info.siblings}
    except Exception as e:
        print(f"Не удалось получить список файлов {repo_id} с Hugging Face ({type(e).__name__}: {e}); проверка размеров файлов пропущена.")
        return None


def missing_or_wrong(path, index):
    bad = []
    for fname, size in index.items():
        p = os.path.join(path, fname)
        if not os.path.isfile(p) or (size is not None and os.path.getsize(p) != size):
            bad.append(fname)
    return bad


os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
os.makedirs(MODEL_PATH_RESOLVED, exist_ok=True)

_du = shutil.disk_usage(PROJECT_DIR)
print(f"Диск ({PROJECT_DIR}): всего {_du.total / GIB:.1f} ГиБ, занято {_du.used / GIB:.1f}, свободно {_du.free / GIB:.1f} ГиБ")
print("(это данные файловой системы; соответствие квоте проекта 50 ГБ - НЕ ПРОВЕРЕНО)")

REPO_INDEX = fetch_repo_index(MODEL)
REPO_SIZE_GIB = None if REPO_INDEX is None else sum((s or 0) for s in REPO_INDEX.values()) / GIB
if REPO_SIZE_GIB is not None:
    print(f"Размер репозитория {MODEL} на Hugging Face: {REPO_SIZE_GIB:.1f} ГиБ")

if REPO_INDEX is not None:
    _bad = missing_or_wrong(MODEL_PATH_RESOLVED, REPO_INDEX)
    _need_download = bool(_bad)
else:
    _bad = []
    _need_download = not weights_present(MODEL_PATH_RESOLVED)
    if not _need_download:
        print("Веса лежат в MODEL_PATH; целостность по размерам файлов НЕ ПРОВЕРЕНА (список файлов с Hugging Face получить не удалось).")

if _need_download:
    print("Нужно скачать в MODEL_PATH:", _bad if _bad else "(весов нет)")
    assert HF_REACHABLE is True, (
        "Весов в MODEL_PATH нет (или они неполные), а Hugging Face недоступен. Положите веса в "
        f"{MODEL_PATH_RESOLVED} вручную (инструкция в начале раздела 1) и перезапустите ячейку.")
    assert _du.free / GIB >= MIN_FREE_DISK_GB, (
        f"Свободно на диске {_du.free / GIB:.1f} ГиБ < MIN_FREE_DISK_GB={MIN_FREE_DISK_GB}. "
        "Лимит хранилища проекта 50 ГБ (включая пакеты): освободите место и перезапустите ячейку. Веса не скачиваются.")
    if REPO_INDEX is not None:
        _remaining = sum((REPO_INDEX[f] or 0) for f in _bad) / GIB
        assert _du.free / GIB >= _remaining + 3, f"Для докачки нужно ~{_remaining:.1f} ГиБ плюс запас 3 ГиБ, свободно {_du.free / GIB:.1f} ГиБ."
    from huggingface_hub import snapshot_download
    _t0 = time.time()
    snapshot_download(repo_id=MODEL, local_dir=MODEL_PATH_RESOLVED, cache_dir=os.environ["HF_HUB_CACHE"], max_workers=4)
    print(f"Скачивание завершено за {(time.time() - _t0) / 60:.1f} мин")
    if REPO_INDEX is not None:
        _bad_after = missing_or_wrong(MODEL_PATH_RESOLVED, REPO_INDEX)
        assert not _bad_after, f"После скачивания отсутствуют или неполные файлы: {_bad_after}. Запустите ячейку ещё раз (докачка продолжится)."
else:
    print("Скачивание не требуется: веса уже в MODEL_PATH.")
assert weights_present(MODEL_PATH_RESOLVED), f"В {MODEL_PATH_RESOLVED} нет config.json или файлов весов (*.safetensors / *.bin)."

WEIGHTS_GIB = dir_size_bytes(MODEL_PATH_RESOLVED) / GIB
HF_CACHE_GIB = dir_size_bytes(HF_CACHE_DIR) / GIB
print(f"Размер weights ({MODEL_PATH_RESOLVED}): {WEIGHTS_GIB:.1f} ГиБ")
print(f"Размер .hf_cache ({HF_CACHE_DIR}): {HF_CACHE_GIB:.1f} ГиБ")
if WEIGHTS_GIB + HF_CACHE_GIB > WEIGHTS_CACHE_WARN_GB:
    print("!" * 100)
    print(f"ПРЕДУПРЕЖДЕНИЕ: weights + .hf_cache = {WEIGHTS_GIB + HF_CACHE_GIB:.1f} ГиБ > {WEIGHTS_CACHE_WARN_GB} ГиБ: вероятно, веса лежат в двух копиях.")
    print("   Лимит хранилища проекта 50 ГБ включает пакеты. Лишнюю копию можно удалить ВРУЧНУЮ, например папку модели в кэше:")
    print(f"   {os.path.join(os.environ['HF_HUB_CACHE'], 'models--' + MODEL.replace('/', '--'))}")
    print("!" * 100)

LOCAL_FILES_ONLY_RESOLVED = True if LOCAL_FILES_ONLY is None else bool(LOCAL_FILES_ONLY)
ENV_INFO.update({
    "local_files_only": LOCAL_FILES_ONLY_RESOLVED,
    "weights_gib": round(WEIGHTS_GIB, 2),
    "hf_cache_gib": round(HF_CACHE_GIB, 2),
    "hf_repo_size_gib": None if REPO_SIZE_GIB is None else round(REPO_SIZE_GIB, 2),
    "disk_free_gib_at_check": round(_du.free / GIB, 2),
    "weights_integrity_checked_by_size": REPO_INDEX is not None,
})
print("OK: веса готовы, local_files_only =", LOCAL_FILES_ONLY_RESOLVED)

In [ ]:
# ===================== 1.3 PYJUICE (импортируется кодом репозитория на уровне модуля) =====================
# llada_generate.py, codd_llada.py и codd/model_class.py делают `import pyjuice as juice` сразу при импорте.
# В exp1 путь HMM не вызывается, но импорт нужен. В блокнотах 2-5 заглушка ЗАПРЕЩЕНА.
PYJUICE_MODE = None


def try_import_pyjuice():
    try:
        import pyjuice  # noqa: F401
        return True, None
    except Exception as e:
        for _k in [k for k in sys.modules if k == "pyjuice" or k.startswith("pyjuice.")]:
            del sys.modules[_k]
        return False, e


_ok, _err = try_import_pyjuice()
if not _ok and INSTALL_DEPS and dist_version("pyjuice") is None:
    print("pyjuice не установлен, пробую pip install pyjuice==2.4.1 (torch и CUDA-стек защищены constraints, переустановки не будет)")
    _cpath, _pins = write_pip_constraints()
    _cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-c", _cpath]
    if PIP_FIND_LINKS:
        _cmd += ["--no-index", "--find-links", PIP_FIND_LINKS]
    _cmd += ["pyjuice==2.4.1"]
    _res = subprocess.run(_cmd, capture_output=True, text=True)
    print(_res.stdout[-1500:])
    if _res.returncode != 0:
        print(_res.stderr[-2500:])
    importlib.invalidate_caches()
    _ok, _err = try_import_pyjuice()

if _ok:
    PYJUICE_MODE = "real"
    print("pyjuice импортирован, версия:", dist_version("pyjuice"), "(ожидается 2.4.1)")
else:
    print("pyjuice НЕ импортируется:", type(_err).__name__, _err)
    if not ALLOW_PYJUICE_STUB:
        raise RuntimeError(
            "pyjuice недоступен. Для exp1 (HMM не используется) можно поставить ALLOW_PYJUICE_STUB=True и перезапустить ячейку, "
            "но ТОЛЬКО в exp1: в блокнотах 2-5 заглушка запрещена. Иначе установите pyjuice (см. инструкцию в начале раздела 1).")
    import types
    _stub = types.ModuleType("pyjuice")

    def _forbidden(*args, **kwargs):
        raise RuntimeError("pyjuice - ЗАГЛУШКА (exp1). Путь HMM здесь вызываться не должен.")

    _stub.load = _forbidden
    _stub.compile = _forbidden
    _stub.save = _forbidden
    _stub.queries = types.SimpleNamespace(sample=_forbidden)
    _stub.__is_stub__ = True
    sys.modules["pyjuice"] = _stub
    PYJUICE_MODE = "stub"
    print("=" * 100)
    print("ВНИМАНИЕ: ИСПОЛЬЗУЕТСЯ ЗАГЛУШКА pyjuice. Допустимо ТОЛЬКО в exp1 (use_pc=False). В блокнотах 2-5 заглушка ЗАПРЕЩЕНА.")
    print("=" * 100)
assert PYJUICE_MODE in ("real", "stub")
ENV_INFO["pyjuice"] = dist_version("pyjuice") if PYJUICE_MODE == "real" else "stub"
ENV_INFO["pyjuice_mode"] = PYJUICE_MODE

In [ ]:
# ===================== 1.4 ИМПОРТ КОДА РЕПОЗИТОРИЯ, env.json, config.json =====================
import re

for _p in (os.path.join(CODD_REPO_DIR, "lm-evaluation-harness"), CODD_REPO_DIR):
    if _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)   # репозиторий должен перекрывать любые другие версии (в т.ч. установленный lm_eval)
importlib.invalidate_caches()

from eval.llada.codd_llada import CoddLlada
from eval.llada.modeling_llada import LLaDAModelLM
from eval.llada import llada_generate as ORIG_GEN_MODULE
from eval.llada.llada_generate import llada_diffusion_generate, add_gumbel_noise, get_num_transfer_tokens, _is_codd_model
import lm_eval
from lm_eval.tasks import TaskManager, get_task_dict
from lm_eval.models.huggingface import HFLM
from lm_eval.models.utils import handle_stop_sequences, postprocess_generated_text

_repo_real = os.path.realpath(CODD_REPO_DIR) + os.sep
for _name, _mod in (("eval.llada.llada_generate", ORIG_GEN_MODULE), ("lm_eval", lm_eval)):
    _file = os.path.realpath(_mod.__file__)
    assert _file.startswith(_repo_real), f"{_name} импортирован НЕ из репозитория: {_file} (ожидалась папка {_repo_real}). Проверьте sys.path / посторонние пакеты с таким же именем."
print("Код репозитория импортирован из:", CODD_REPO_DIR)


def git_info(repo):
    def run(args):
        return subprocess.run(["git", "-C", repo] + args, capture_output=True, text=True, timeout=20)
    try:
        head = run(["rev-parse", "HEAD"])
        if head.returncode != 0:
            return {"commit": "unknown (нет git-метаданных)", "dirty": None}
        status = run(["status", "--porcelain"])
        return {"commit": head.stdout.strip(), "dirty": bool(status.stdout.strip())}
    except Exception as e:
        return {"commit": f"unknown ({type(e).__name__})", "dirty": None}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


_pyproject = open(os.path.join(CODD_REPO_DIR, "lm-evaluation-harness", "pyproject.toml"), encoding="utf-8").read()
_m = re.search(r'^version\s*=\s*"([^"]+)"', _pyproject, re.M)
_git = git_info(CODD_REPO_DIR)
ENV_INFO.update({
    "transformers": dist_version("transformers"),
    "huggingface_hub": dist_version("huggingface-hub"),
    "numpy": dist_version("numpy"),
    "lm_eval_version_pyproject": _m.group(1) if _m else None,
    "codd_commit": _git["commit"],
    "codd_dirty": _git["dirty"],
    "llada_generate_py_sha256": sha256_file(os.path.join(CODD_REPO_DIR, "eval", "llada", "llada_generate.py")),
    "common_params_sha256": COMMON_SHA256,
    "exp_params_sha256": EXP_SHA256,
})

ENV_PATH = os.path.join(OUT_DIR, "env.json")
if os.path.exists(ENV_PATH):
    with open(ENV_PATH, encoding="utf-8") as _f:
        _old_env = json.load(_f)
    _prev = _old_env.pop("previous_sessions", [])
    ENV_INFO["previous_sessions"] = _prev + [_old_env]   # история прежних сессий (возобновление после обрыва)
write_json_atomic(ENV_PATH, ENV_INFO)
print("env.json записан:", ENV_PATH)

CONFIG_PATH = os.path.join(OUT_DIR, "config.json")
_config_doc = {"common_params": COMMON_PARAMS, "common_sha256": COMMON_SHA256, "exp_params": EXP_PARAMS, "exp_sha256": EXP_SHA256}
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, encoding="utf-8") as _f:
        _old_cfg = json.load(_f)
    if _old_cfg.get("exp_sha256") != EXP_SHA256:
        _keys = set(_old_cfg.get("exp_params", {})) | set(EXP_PARAMS)
        _diff = {k: (_old_cfg.get("exp_params", {}).get(k), EXP_PARAMS.get(k)) for k in _keys
                 if _old_cfg.get("exp_params", {}).get(k) != EXP_PARAMS.get(k)}
        raise RuntimeError(
            f"В {OUT_DIR} уже лежат результаты с ДРУГИМ конфигом (параметр: (было, стало)): {_diff}. "
            "Чтобы не смешать результаты, верните прежние значения или укажите другой OUT_DIR.")
    print("config.json совпадает с текущим конфигом - возобновление допустимо.")
else:
    write_json_atomic(CONFIG_PATH, _config_doc)
    print("config.json записан:", CONFIG_PATH)
print("codd_commit:", ENV_INFO["codd_commit"], "| dirty:", ENV_INFO["codd_dirty"])

## 2. Загрузка модели (один раз)

Модель грузится **один раз** через `CoddLlada.from_pretrained` без `pc_model_id` (как `eval.py` для режима с PC, только без PC): `pc_model` остаётся `None`, forward возвращает логиты базовой LLaDA. Класс `PlainView` даёт ту же модель как "обычную" (вызов `LLaDAModelLM.forward`), чтобы прогнать базовую ветку оригинальной функции без второй загрузки весов.

После загрузки сверяются sha256 исходников модели (`modeling_llada.py`, `configuration_llada.py`) из репозитория и из снапшота весов: это единственная проверка того, что код `CoddLlada` (репозиторий) и код `AutoModel(trust_remote_code)` (снапшот на Hugging Face) совпадают. Совпадение исходников не доказывает бит-в-бит одинаковый вывод (зависит от dtype, версий библиотек и железа).

In [ ]:
# ===================== 2.1 ЗАГРУЗКА МОДЕЛИ =====================
if globals().get("model") is not None:
    print("Модель уже загружена в этом ядре - повторная загрузка пропущена.")
else:
    _t0 = time.time()
    model = CoddLlada.from_pretrained(
        MODEL_PATH_RESOLVED,
        torch_dtype=TORCH_DTYPE,
        local_files_only=LOCAL_FILES_ONLY_RESOLVED,
        attn_implementation=ATTN_IMPLEMENTATION,
        device_map="cuda",
    )
    ENV_INFO["model_load_seconds"] = round(time.time() - _t0, 1)
    print(f"Модель загружена за {time.time() - _t0:.0f} c")

model.eval()
for _p in model.parameters():
    _p.requires_grad_(False)
tokenizer = model.tokenizer

# --- проверки ---
assert tokenizer is not None, "model.tokenizer is None: токенизатор не загрузился"
assert model.pc_model is None, "pc_model должен быть None: в exp1 HMM не используется и не должен загружаться"
assert model.codd_config.mask_id == MASK_ID, f"mask_id модели {model.codd_config.mask_id} != {MASK_ID}"
_emb = model.get_input_embeddings().weight
assert _emb.shape[0] > MASK_ID, f"Размер словаря {_emb.shape[0]} не содержит mask_id={MASK_ID}"
assert next(model.parameters()).dtype == TORCH_DTYPE, f"ожидается {DTYPE}, а параметры модели в {next(model.parameters()).dtype}"
print("dtype модели:", next(model.parameters()).dtype, "(DTYPE из конфига:", DTYPE + ")")
assert next(model.parameters()).device.type == "cuda", "модель должна быть на GPU"
assert not model.training, "модель должна быть в eval-режиме"
print("mask_id", MASK_ID, "-> токен:", tokenizer.convert_ids_to_tokens(MASK_ID), "| eos_token_id:", tokenizer.eos_token_id)
print(f"Память GPU занята: {torch.cuda.memory_allocated() / 1024 ** 3:.1f} ГБ")


class PlainView:
    # Та же модель, но как "обычная": вызов базового forward без CoDD-аргументов. Атрибута pc_model нет специально:
    # _is_codd_model(PlainView(...)) == False, поэтому оригинальная функция идёт по базовой ветке (model(x).logits).
    def __init__(self, m):
        self._m = m

    @property
    def device(self):
        return self._m.device

    def __call__(self, x):
        return LLaDAModelLM.forward(self._m, x)


PLAIN = PlainView(model)
assert _is_codd_model(model) is True and _is_codd_model(PLAIN) is False

In [ ]:
# ===================== 2.2 СВЕРКА ИСХОДНИКОВ МОДЕЛИ (sha256) =====================
# Вместо загрузки второй модели: сравниваем файлы репозитория с одноимёнными файлами из снапшота весов.
def find_snapshot_file(fname):
    if os.path.isdir(MODEL_PATH_RESOLVED):
        p = os.path.join(MODEL_PATH_RESOLVED, fname)
        return (p, "локальная папка весов") if os.path.isfile(p) else (None, f"в папке весов {MODEL_PATH_RESOLVED} нет файла {fname}")
    try:
        from huggingface_hub import try_to_load_from_cache
        p = try_to_load_from_cache(MODEL, fname)
        if isinstance(p, str) and os.path.isfile(p):
            return p, "кэш Hugging Face"
    except Exception:
        pass
    if not LOCAL_FILES_ONLY_RESOLVED and HF_REACHABLE is True:
        try:
            from huggingface_hub import hf_hub_download
            return hf_hub_download(MODEL, fname), "загрузка с Hugging Face"
        except Exception as e:
            return None, f"не удалось получить {fname} с Hugging Face ({type(e).__name__})"
    return None, f"файла {fname} нет ни в кэше Hugging Face, ни рядом с весами, а сеть недоступна/запрещена"


def sha256_text_normalized(path):
    with open(path, "rb") as f:
        data = f.read().replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()


MODEL_CODE_CHECK = {}
for _fname in ("modeling_llada.py", "configuration_llada.py"):
    _local = os.path.join(CODD_REPO_DIR, "eval", "llada", _fname)
    _snap, _how = find_snapshot_file(_fname)
    if _snap is None:
        MODEL_CODE_CHECK[_fname] = {"verdict": "НЕ ПРОВЕРЕНО", "reason": _how}
        print(f"{_fname}: НЕ ПРОВЕРЕНО - {_how}")
        continue
    _a, _b = sha256_file(_local), sha256_file(_snap)
    _same_norm = sha256_text_normalized(_local) == sha256_text_normalized(_snap)
    MODEL_CODE_CHECK[_fname] = {
        "verdict": "MATCH" if _a == _b else "MISMATCH", "repo_sha256": _a, "snapshot_sha256": _b,
        "snapshot_source": _how, "equal_after_newline_normalization": _same_norm,
    }
    print(f"{_fname}: {MODEL_CODE_CHECK[_fname]['verdict']} (источник снапшота: {_how}); repo={_a[:16]}... snapshot={_b[:16]}...")
    if _a != _b and _same_norm:
        print("   (файлы различаются только переводами строк)")

ENV_INFO["model_code_check"] = MODEL_CODE_CHECK
write_json_atomic(ENV_PATH, ENV_INFO)
_bad = [k for k, v in MODEL_CODE_CHECK.items() if v["verdict"] == "MISMATCH"]
if _bad and not ALLOW_MODEL_CODE_MISMATCH:
    raise RuntimeError(
        f"Исходники модели из репозитория CoDD отличаются от файлов в снапшоте весов: {_bad}. "
        "Значит, CoddLlada и AutoModel(trust_remote_code) могут вести себя по-разному. Остановка до решения автора проекта "
        "(чтобы продолжить осознанно: ALLOW_MODEL_CODE_MISMATCH=True и отметить это в отчёте).")
if all(v["verdict"] == "НЕ ПРОВЕРЕНО" for v in MODEL_CODE_CHECK.values()):
    print("ИТОГ: эквивалентность кода CoddLlada и remote-кода весов НЕ ПРОВЕРЕНА (см. причины выше).")

## 3. Данные и задачи (с хэшем)

Задача `gsm8k` берётся из `lm-evaluation-harness` репозитория (`TaskManager` / `get_task_dict`, как в `eval.py`). Как и `simple_evaluate` при `--num_shot 0`, число примеров в промпте принудительно ставится в 0. Контексты строит `task.build_all_requests(...)` репозитория с `apply_chat_template=True` и `system_instruction="You are a helpful assistant."`; шаблон чата, токенизацию и декодирование даёт `HFLM(pretrained=model, tokenizer=...)`, как в `LladaEvalHarness`. Берутся первые `N_TASKS` задач test-выборки в порядке датасета; `task_id` = индекс в выборке.

Хэш списка задач: `sha256(json.dumps(ids, separators=(",",":")).encode("utf-8"))` - печатается и сохраняется в `tasks.json` (поле `ids_sha256`). У всех пяти блокнотов он должен совпасть (для ids 0..99 это одно и то же значение). Дополнительно, справочно, считается sha256 текстов вопросов.

Если Hugging Face недоступен, датасет берётся из локальных файлов `DATA_DIR` (подмена `datasets.load_dataset` на время создания задачи; **НЕ ПРОВЕРЕНО**).

In [ ]:
# ===================== 3.1 ЗАДАЧА GSM8K, МОДЕЛЬ-ОБЁРТКА HFLM, КОНТЕКСТЫ =====================
import copy
import glob
import datasets
from unittest import mock


def build_local_gsm8k(data_dir):
    # Локальный запасной вариант датасета: parquet (структура openai/gsm8k) или jsonl с полями question, answer.
    def find(split):
        patterns = [f"{split}.jsonl", f"{split}.json", f"{split}.parquet",
                    os.path.join("main", f"{split}.jsonl"), os.path.join("main", f"{split}-*.parquet"), f"{split}-*.parquet"]
        for pat in patterns:
            hits = sorted(glob.glob(os.path.join(data_dir, pat)))
            if len(hits) == 1:
                return hits[0]
            if len(hits) > 1:
                raise RuntimeError(f"В {data_dir} несколько файлов для сплита {split}: {hits}. Оставьте один.")
        return None

    splits = {}
    for split in ("train", "test"):
        path = find(split)
        if path is None:
            continue
        ds = datasets.Dataset.from_parquet(path) if path.endswith(".parquet") else datasets.Dataset.from_json(path)
        assert {"question", "answer"} <= set(ds.column_names), f"{path}: нужны поля question и answer, есть {ds.column_names}"
        splits[split] = ds
        print(f"локальный датасет: {split} <- {path} ({len(ds)} примеров)")
    if "test" not in splits:
        raise FileNotFoundError(f"В {data_dir} не найден test-сплит GSM8K (см. инструкцию в разделе 1).")
    if "train" not in splits:
        print("ВНИМАНИЕ: train-сплита нет; вместо него подставлен test. На 0-shot это не влияет (примеры в промпт не берутся).")
        splits["train"] = splits["test"]
    return datasets.DatasetDict(splits)


def load_task():
    task_manager = TaskManager()
    task_dict = get_task_dict([TASK_NAME], task_manager=task_manager)
    assert len(task_dict) == 1, f"ожидалась одна задача {TASK_NAME}, получено: {list(task_dict)}"
    return next(iter(task_dict.values()))


try:
    task = load_task()
    DATA_SOURCE = "Hugging Face datasets (кэш или сеть)"
except Exception as _e_hf:
    print(f"Загрузка датасета через lm_eval/Hugging Face не удалась: {type(_e_hf).__name__}: {_e_hf}")
    print(f"Пробую локальные файлы в {DATA_DIR} ...")
    try:
        _local_ds = build_local_gsm8k(DATA_DIR)
    except Exception as _e_local:
        raise RuntimeError(
            f"Не удалось получить GSM8K ни через Hugging Face ({type(_e_hf).__name__}: {_e_hf}), "
            f"ни из локальных файлов {DATA_DIR} ({type(_e_local).__name__}: {_e_local}). "
            "Положите датасет в DATA_DIR (инструкция в разделе 1).") from _e_local
    with mock.patch.object(datasets, "load_dataset", lambda *args, **kwargs: _local_ds):
        task = load_task()
    DATA_SOURCE = f"локальные файлы {DATA_DIR}"
print("Источник датасета:", DATA_SOURCE)

# --- как simple_evaluate при --num_shot 0 ---
task.set_config(key="num_fewshot", value=NUM_FEWSHOT)
assert task.get_config("num_fewshot") == NUM_FEWSHOT == 0
assert task.OUTPUT_TYPE == "generate_until", f"ожидался generate_until, а не {task.OUTPUT_TYPE}"
_gk = task.get_config("generation_kwargs")
assert _gk.get("do_sample") is False and float(_gk.get("temperature", 0.0)) == TEMPERATURE == 0.0, f"generation_kwargs задачи не жадные: {_gk}"
_n_docs = len(task.eval_docs)
print(f"Документов в {DATASET_SPLIT}-выборке: {_n_docs}", "" if _n_docs == 1319 else "(ВНИМАНИЕ: у стандартного GSM8K test 1319 задач)")
assert _n_docs >= N_TASKS, f"в выборке {_n_docs} < N_TASKS={N_TASKS}"
assert {f.name for f in task._filters} == {"strict-match", "flexible-extract"}, f"неожиданные фильтры задачи: {[f.name for f in task._filters]}"

# --- обёртка HFLM (как LladaEvalHarness): токенизация, chat template, декодирование ---
lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=1)
print("HFLM backend:", lm.backend, "| max_length (HFLM):", lm.max_length, "| tokenizer_name:", lm.tokenizer_name)

task.build_all_requests(
    limit=N_TASKS,
    rank=0,
    world_size=1,
    cache_requests=False,
    rewrite_requests_cache=False,
    system_instruction=SYSTEM_INSTRUCTION,
    apply_chat_template=True,
    fewshot_as_multiturn=False,
    chat_template=lm.apply_chat_template,
    tokenizer_name=lm.tokenizer_name,
)
INSTANCES = list(task.instances)
assert len(INSTANCES) == N_TASKS, f"получено {len(INSTANCES)} запросов вместо {N_TASKS}"
for _i, _inst in enumerate(INSTANCES):
    assert _inst.doc_id == _i, f"порядок задач нарушен: позиция {_i}, doc_id {_inst.doc_id}"
    assert _inst.request_type == "generate_until" and _inst.repeats == 1
TASK_IDS = [inst.doc_id for inst in INSTANCES]
assert TASK_IDS == list(range(N_TASKS))

# --- условия остановки декодирования ("until") берутся из задачи и от GEN_LENGTH не зависят ---
_gen_kwargs = copy.deepcopy(INSTANCES[0].args[1])
UNTIL = handle_stop_sequences(_gen_kwargs.pop("until", None), eos=lm.tok_decode(lm.eot_token_id, skip_special_tokens=False))
print("until (стоп-последовательности из задачи + EOS):", UNTIL)
assert all(inst.args[1] == INSTANCES[0].args[1] for inst in INSTANCES), "generation_kwargs различаются между задачами"

# --- токенизация промптов (как HFLM.generate_until, но без усечения: вместо него assert) ---
PROMPT_IDS = []
for _inst in INSTANCES:
    _ids, _ = lm.tok_batch_encode([_inst.args[0]], left_truncate_len=None, truncation=False)
    assert _ids.shape[0] == 1
    PROMPT_IDS.append(_ids[0].tolist())
    assert MASK_ID not in PROMPT_IDS[-1], "в промпте встретился mask_id"

# --- ограничение по длине: от GEN_LENGTH зависит ТОЛЬКО это (проверяем оба допустимых значения) ---
_limits = [lm.max_length]
if hasattr(model.config, "max_sequence_length") and model.config.max_sequence_length:
    _limits.append(int(model.config.max_sequence_length))
MODEL_MAX_LEN = min(_limits)
_max_prompt = max(len(p) for p in PROMPT_IDS)
for _gl in sorted({256, 512, GEN_LENGTH}):
    assert _max_prompt + _gl <= MODEL_MAX_LEN, (
        f"Самый длинный промпт ({_max_prompt} ток.) + GEN_LENGTH={_gl} > лимита длины модели {MODEL_MAX_LEN}. "
        "В исходном харнессе это привело бы к молчаливому усечению промпта слева; здесь усечения нет, поэтому остановка.")
print(f"Промпты: длина {min(len(p) for p in PROMPT_IDS)}..{_max_prompt} токенов; лимит модели {MODEL_MAX_LEN}; GEN_LENGTH={GEN_LENGTH} помещается (проверено и для 256, и для 512).")

print("-" * 80)
print("ПРОМПТ задачи 0 (как видит модель):")
print(INSTANCES[0].args[0])
print("-" * 80)
print("токены начала промпта:", PROMPT_IDS[0][:12], "->", repr(tokenizer.decode(PROMPT_IDS[0][:12])))

In [ ]:
# ===================== 3.2 tasks.json И ХЭШ СПИСКА ЗАДАЧ =====================
IDS_SHA256 = hashlib.sha256(json.dumps(TASK_IDS, separators=(",", ":")).encode("utf-8")).hexdigest()
QUESTIONS_SHA256 = hashlib.sha256(json.dumps([inst.doc["question"] for inst in INSTANCES], ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()


def gold_final(answer_text):
    m = re.search(r"####\s*([^\n]+)", answer_text)
    return m.group(1).strip() if m else None


TASKS_DOC = {
    "dataset": DATASET, "split": DATASET_SPLIT, "n_tasks": N_TASKS, "data_source": DATA_SOURCE,
    "system_instruction": SYSTEM_INSTRUCTION, "num_fewshot": NUM_FEWSHOT, "chat_template_applied": True,
    "ids_sha256": IDS_SHA256,
    "ids_sha256_definition": 'sha256(json.dumps(ids, separators=(",",":")).encode("utf-8")), ids = [0..N_TASKS-1]',
    "questions_sha256": QUESTIONS_SHA256,
    "ids": TASK_IDS,
    "tasks": [
        {"task_id": inst.doc_id, "question": inst.doc["question"], "gold_final": gold_final(inst.doc["answer"]),
         "prompt_sha256": hashlib.sha256(inst.args[0].encode("utf-8")).hexdigest()}
        for inst in INSTANCES
    ],
}
assert all(t["gold_final"] is not None for t in TASKS_DOC["tasks"]), "у некоторых задач нет '#### ответ' в эталоне"

TASKS_PATH = os.path.join(OUT_DIR, "tasks.json")
if os.path.exists(TASKS_PATH):
    with open(TASKS_PATH, encoding="utf-8") as _f:
        _old = json.load(_f)
    assert _old["ids_sha256"] == IDS_SHA256, "список id отличается от сохранённого в tasks.json - возобновление небезопасно"
    assert _old["questions_sha256"] == QUESTIONS_SHA256, "тексты вопросов отличаются от сохранённых в tasks.json (другая версия датасета?)"
    _old_prompts = {t["task_id"]: t["prompt_sha256"] for t in _old["tasks"]}
    _new_prompts = {t["task_id"]: t["prompt_sha256"] for t in TASKS_DOC["tasks"]}
    assert _old_prompts == _new_prompts, "промпты отличаются от сохранённых в tasks.json (другой шаблон чата/токенизатор?) - возобновление небезопасно"
    print("tasks.json совпадает с текущими задачами и промптами - возобновление допустимо.")
else:
    write_json_atomic(TASKS_PATH, TASKS_DOC)
    print("tasks.json записан:", TASKS_PATH)

print("=" * 80)
print("sha256 списка id задач:", IDS_SHA256)
print("(справочно) sha256 текстов вопросов:", QUESTIONS_SHA256)
print("=" * 80)

## 4. Функции: копия генерации с журналом, оценка, кэш и возобновление

- **`llada_generate_logged`** - построчная копия `llada_diffusion_generate` из `eval/llada/llada_generate.py` (коммит d9760f4). Вычисления не изменены; добавлены только (1) `assert use_pc is False`, (2) журнал шагов (какие позиции раскрыты, какие токены, какая была уверенность), (3) `torch.topk` возвращает и значения (оригинал их отбрасывал), (4) убран `tqdm`. Функции `add_gumbel_noise`, `get_num_transfer_tokens`, `_is_codd_model` импортируются из репозитория, а не копируются. Совпадение токенов с оригиналом проверяет дымовой тест (раздел 5).
- **Оценка** ответа: фильтры и метрика задачи `gsm8k` из репозитория (`task._filters`, `task.process_results`), а не свой код. Основной `correct` - `exact_match` под фильтром `flexible-extract` (как `eval/utils.py::parse_results`); `correct_strict` (под `strict-match`) пишется дополнительно.
- **Кэш**: после КАЖДОЙ задачи сначала дописывается блок строк steplog, затем строка результата (она - маркер завершения), с `flush` и `fsync`. При возобновлении готовые `task_id` пропускаются, а строки steplog задач без результата удаляются.
- **Файлы**: `results/exp1_llada_default/{remasking}_steps{S}_seed0.jsonl` и `steplog_{remasking}_steps{S}_seed0.jsonl`. Поля `delta_words`, `delta_order` (заготовки под разделение вкладов) в exp1 всегда `null`, `pc_active_steps` всегда `0`.

In [ ]:
# ===================== 4.1 КОПИЯ llada_diffusion_generate С ЖУРНАЛОМ ШАГОВ =====================
import math
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm


@torch.no_grad()
def llada_generate_logged(
    model, prompt, num_steps=128, gen_length=128, block_length=128,
    temperature=0., cfg_scale=0., remasking='low_confidence', mask_id=126336,
    use_pc=False, pc_temperature=None, pc_frac=None, reverse_frac=None,
):
    # Копия llada_diffusion_generate (CoDD/eval/llada/llada_generate.py, коммит d9760f4) + журнал шагов.
    # Отличия от оригинала (на вычисления не влияют): assert use_pc is False; нет tqdm; steplog; topk сохраняет значения.
    assert use_pc is False, "exp1: HMM не используется (use_pc=False)"
    assert prompt.dim() == 2 and prompt.shape[0] == 1, "ожидается промпт формы (1, L)"
    is_codd = _is_codd_model(model)

    x = torch.full((1, prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(model.device)
    x[:, :prompt.shape[1]] = prompt.clone()

    prompt_index = (x != mask_id)

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length

    assert num_steps % num_blocks == 0
    steps = num_steps // num_blocks

    steplog = []
    for num_block in range(num_blocks):
        start_idx = prompt.shape[1] + num_block * block_length
        end_idx = prompt.shape[1] + (num_block + 1) * block_length

        block_mask_index = (x[:, start_idx:end_idx] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps)

        for i in range(steps):
            mask_index = (x == mask_id)
            n_masked_block_before = int(mask_index[0, start_idx:end_idx].sum().item())

            # ---------- forward pass ----------
            if is_codd:
                # CoDD.forward() handles CFG + PC modification internally
                pc_block_range = (start_idx, end_idx) if use_pc else None
                output = model(
                    x,
                    pc_block_range=pc_block_range,
                    use_pc=use_pc,
                    cfg_scale=cfg_scale,
                    prompt_index=prompt_index if cfg_scale > 0. else None,
                    pc_temperature=pc_temperature,
                    pc_frac=pc_frac,
                    reverse_frac=reverse_frac,
                )
                logits = output.logits
            else:
                # Plain base model path (backward compatible)
                if cfg_scale > 0.:
                    un_x = x.clone()
                    un_x[prompt_index] = mask_id
                    x_ = torch.cat([x, un_x], dim=0)
                    logits = model(x_).logits
                    logits, un_logits = torch.chunk(logits, 2, dim=0)
                    logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
                else:
                    logits = model(x).logits

            # ---------- sampling ----------
            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)

            if remasking == 'low_confidence':
                p = F.softmax(logits.to(torch.float64), dim=-1)
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
            elif remasking == 'random':
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            elif remasking == 'margin':
                probs = torch.softmax(logits_with_noise, dim=-1)
                top_probs, _ = torch.topk(probs, k=2, dim=-1)
                x0_p = top_probs[..., 0] - top_probs[..., 1]
            elif remasking == 'entropy':
                probs = torch.softmax(logits_with_noise, dim=-1)
                epsilon = 1e-10
                log_probs = torch.log(probs + epsilon)
                x0_p = torch.sum(probs * log_probs, dim=-1)
            else:
                raise NotImplementedError(remasking)

            x0_p[:, end_idx:] = -np.inf

            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            for j in range(confidence.shape[0]):
                num_tokens = num_transfer_tokens[j, i].item()
                if num_tokens > 0:
                    conf_vals, select_index = torch.topk(confidence[j], k=num_tokens)
                    x[j, select_index] = x0[j, select_index]
                    pos_abs = select_index.tolist()
                    steplog.append({
                        "block": num_block,
                        "step_in_block": i,
                        "global_step": num_block * steps + i,
                        "n_masked_block_before": n_masked_block_before,
                        "positions_abs": pos_abs,
                        "positions_rel": [p_ - prompt.shape[1] for p_ in pos_abs],
                        "token_ids": x0[j, select_index].tolist(),
                        "confidence": [float(c) for c in conf_vals.float().tolist()],
                    })

    return x, steplog


print("llada_generate_logged определена (копия оригинала + журнал).")

In [ ]:
# ===================== 4.2 ДЕКОДИРОВАНИЕ И ПРОВЕРКА ОТВЕТА (код репозитория) =====================
FILTER_NAMES = ("strict-match", "flexible-extract")


def score_response(inst, text):
    # Ровно то, что делает evaluator.evaluate: resps -> task.apply_filters (фильтры покомпонентные, поэтому
    # применение к одному запросу эквивалентно применению ко всем) -> task.process_results(doc, [filtered_resps]).
    inst.resps = [text]
    inst.filtered_resps = {}
    for filter_ensemble in task._filters:
        filter_ensemble.apply([inst])
    scores, preds = {}, {}
    for name in FILTER_NAMES:
        assert name in inst.filtered_resps, f"нет результата фильтра {name}: {list(inst.filtered_resps)}"
        pred = inst.filtered_resps[name]
        assert isinstance(pred, str), f"ожидалась строка после фильтра {name}, получено {type(pred).__name__}: {pred!r}"
        metrics = task.process_results(inst.doc, [pred])
        assert "exact_match" in metrics, f"process_results не вернул exact_match: {metrics}"
        value = float(metrics["exact_match"])
        assert value in (0.0, 1.0), f"exact_match должен быть 0 или 1, получено {value}"
        scores[name] = value
        preds[name] = pred
    return scores, preds


def decode_generation(gen_token_ids):
    # Как HFLM.generate_until: tok_decode(cont_toks) -> postprocess_generated_text по until.
    text_raw = lm.tok_decode(gen_token_ids)
    return postprocess_generated_text(generation=text_raw, stop=UNTIL, think_end_token=None)


def run_generation(inst, remasking, num_steps, gen_length=None, block_length=None):
    # Одна генерация для задачи; возвращает (x, steplog, prompt_len, секунды).
    gen_length = GEN_LENGTH if gen_length is None else gen_length
    block_length = BLOCK_LENGTH if block_length is None else block_length
    prompt = torch.tensor([PROMPT_IDS[inst.doc_id]], dtype=torch.long, device=model.device)
    torch.cuda.synchronize()
    t0 = time.time()
    x, steplog = llada_generate_logged(
        model, prompt, num_steps=num_steps, gen_length=gen_length, block_length=block_length,
        temperature=TEMPERATURE, cfg_scale=0.0, remasking=remasking, mask_id=MASK_ID, use_pc=False)
    torch.cuda.synchronize()
    return x, steplog, prompt.shape[1], time.time() - t0


def process_task(inst, remasking, num_steps):
    task_id = inst.doc_id
    x, steplog, prompt_len, sec = run_generation(inst, remasking, num_steps)
    gen_ids = x[0, prompt_len:].tolist()
    assert len(gen_ids) == GEN_LENGTH
    assert MASK_ID not in gen_ids, f"задача {task_id}: в ответе остались маски (mask_id={MASK_ID}) - несостыковка, остановка"
    assert len(steplog) == num_steps, f"задача {task_id}: в журнале {len(steplog)} шагов вместо {num_steps}"
    text = decode_generation(gen_ids)
    scores, preds = score_response(inst, text)
    row = {
        "task_id": task_id,
        "seed": SEED_USED,
        "correct": int(scores["flexible-extract"] == 1.0),
        "prediction": preds["flexible-extract"],
        "num_steps": num_steps,
        "remasking": remasking,
        "pc_active_steps": 0,
        "delta_words": None,
        "delta_order": None,
        # --- дополнительные поля (не входят в обязательный формат) ---
        "correct_strict": int(scores["strict-match"] == 1.0),
        "prediction_strict": preds["strict-match"],
        "gold": gold_final(inst.doc["answer"]),
        "generation": text,
        "gen_length": GEN_LENGTH,
        "prompt_tokens": prompt_len,
        "sec": round(sec, 3),
    }
    step_rows = [
        {"task_id": task_id, "seed": SEED_USED, "remasking": remasking, "num_steps": num_steps,
         **rec, "delta_words": None, "delta_order": None, "pc_active_steps": 0}
        for rec in steplog
    ]
    return row, step_rows


print("Функции оценки определены.")

In [ ]:
# ===================== 4.3 КЭШ И ВОЗОБНОВЛЕНИЕ (jsonl) =====================
REQUIRED_ROW_KEYS = ("task_id", "seed", "correct", "prediction", "num_steps", "remasking", "pc_active_steps", "delta_words", "delta_order")


def result_path(remasking, num_steps):
    return os.path.join(OUT_DIR, f"{remasking}_steps{num_steps}_seed{SEED_USED}.jsonl")


def steplog_path(remasking, num_steps):
    return os.path.join(OUT_DIR, f"steplog_{remasking}_steps{num_steps}_seed{SEED_USED}.jsonl")


def read_jsonl_tolerant(path):
    # Читает только полные строки (оканчивающиеся на \n). Оборванный хвост (обрыв сессии во время записи) отбрасывается
    # и физически обрезается; повреждённая ПОЛНАЯ строка - ошибка (автоматически не чиним).
    if not os.path.exists(path):
        return []
    with open(path, "rb") as f:
        data = f.read()
    cut = data.rfind(b"\n") + 1
    tail = data[cut:]
    if tail:
        if tail.strip():
            print(f"ВНИМАНИЕ: {path}: оборванная последняя строка ({len(tail)} байт) отброшена")
        with open(path, "r+b") as f:
            f.truncate(cut)
            f.flush()
            os.fsync(f.fileno())
    rows = []
    for line_no, raw in enumerate(data[:cut].split(b"\n"), start=1):
        if not raw.strip():
            continue
        try:
            rows.append(json.loads(raw.decode("utf-8")))
        except Exception as e:
            raise RuntimeError(f"{path}: строка {line_no} повреждена ({type(e).__name__}: {e}). Исправьте или удалите файл вручную.") from e
    return rows


def append_jsonl(path, records):
    payload = "".join(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n" for r in records)
    with open(path, "a", encoding="utf-8") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())


def repair_steplog(path, done_ids):
    # Удаляет из steplog строки задач, у которых нет строки результата (обрыв между записью steplog и результата).
    rows = read_jsonl_tolerant(path)
    keep = [r for r in rows if r["task_id"] in done_ids]
    if len(keep) != len(rows):
        dropped = sorted({r["task_id"] for r in rows if r["task_id"] not in done_ids})
        print(f"steplog {os.path.basename(path)}: удалены строки незавершённых задач {dropped}")
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for r in keep:
                f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    return keep


def validate_result_row(row, remasking, num_steps):
    missing = [k for k in REQUIRED_ROW_KEYS if k not in row]
    assert not missing, f"в строке результата нет полей {missing}: {row.get('task_id')}"
    assert row["remasking"] == remasking and row["num_steps"] == num_steps, f"строка не из этой комбинации: {row['remasking']}/{row['num_steps']}"
    assert row["seed"] == SEED_USED and row["pc_active_steps"] == 0 and row["delta_words"] is None and row["delta_order"] is None
    assert row["correct"] in (0, 1) and isinstance(row["prediction"], str) and isinstance(row["task_id"], int)
    assert row.get("gen_length", GEN_LENGTH) == GEN_LENGTH, f"в файле результаты с gen_length={row.get('gen_length')}, а GEN_LENGTH={GEN_LENGTH}"


def run_combo(remasking, num_steps, instances_to_run, deadline=None):
    # Возвращает "done" (все задачи комбинации готовы) или "stopped" (сработал deadline = момент time.time(), MAX_RUNTIME_MIN).
    # Остановка происходит только ПОСЛЕ полностью записанной задачи (steplog, затем результат, fsync), поэтому кэш всегда согласован.
    rpath, spath = result_path(remasking, num_steps), steplog_path(remasking, num_steps)
    rows = read_jsonl_tolerant(rpath)
    for r in rows:
        validate_result_row(r, remasking, num_steps)
    ids = [r["task_id"] for r in rows]
    assert len(ids) == len(set(ids)), f"{rpath}: повторяющиеся task_id"
    done = set(ids)
    repair_steplog(spath, done)
    todo = [inst for inst in instances_to_run if inst.doc_id not in done]
    print(f"[{remasking}, steps={num_steps}] уже готово {len(done)}, осталось {len(todo)}")
    for inst in tqdm(todo, desc=f"{remasking}/{num_steps}"):
        row, step_rows = process_task(inst, remasking, num_steps)
        append_jsonl(spath, step_rows)   # 1) журнал шагов
        append_jsonl(rpath, [row])       # 2) строка результата = маркер завершения задачи
        torch.cuda.empty_cache()         # между задачами возвращаем кэш аллокатора (на 24 ГБ запас небольшой)
        if deadline is not None and time.time() >= deadline:
            print(f"[{remasking}, steps={num_steps}] остановка по MAX_RUNTIME_MIN после задачи {inst.doc_id}")
            return "stopped"
    return "done"


def validate_combo(remasking, num_steps, expected_ids):
    rows = read_jsonl_tolerant(result_path(remasking, num_steps))
    for r in rows:
        validate_result_row(r, remasking, num_steps)
    ids = [r["task_id"] for r in rows]
    assert sorted(ids) == sorted(expected_ids), f"{remasking}/{num_steps}: ожидались task_id {len(expected_ids)} шт., в файле {len(ids)} (повторы или пропуски)"
    steps_rows = read_jsonl_tolerant(steplog_path(remasking, num_steps))
    per_task = {}
    for s in steps_rows:
        per_task.setdefault(s["task_id"], []).append(s)
    assert set(per_task) == set(ids), f"{remasking}/{num_steps}: steplog и результаты содержат разные task_id"
    for tid, recs in per_task.items():
        assert len(recs) == num_steps, f"{remasking}/{num_steps}, задача {tid}: в steplog {len(recs)} шагов вместо {num_steps}"
        assert [r["global_step"] for r in recs] == list(range(num_steps)), f"{remasking}/{num_steps}, задача {tid}: нарушен порядок шагов"
    return rows


def bootstrap_ci(values, n_boot=None, seed=None, alpha=0.05):
    # Percentile bootstrap по задачам: ресэмплируем задачи с возвратом, считаем долю верных.
    n_boot = N_BOOTSTRAP if n_boot is None else n_boot
    seed = BOOTSTRAP_SEED if seed is None else seed
    v = np.asarray(values, dtype=np.float64)
    assert v.ndim == 1 and v.size > 0, "пустой набор значений"
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, v.size, size=(n_boot, v.size))
    means = v[idx].mean(axis=1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(v.mean()), float(lo), float(hi)


print("Функции кэша, возобновления и bootstrap определены.")
print("Файлы результатов будут лежать в:", OUT_DIR)

## 5. Дымовой тест на GPU (3 задачи, мало шагов)

Параметры теста: 3 первые задачи, `num_steps=8`, `gen_length=64`, `block_length=32`, оба варианта remasking. Для каждой пары (remasking, задача) сравниваются токены:

1. копия, прогон 1 и прогон 2 (**детерминизм**: если не совпали, это недетерминизм GPU, а не логика);
2. копия на `CoddLlada` (`use_pc=False`) и копия на `PlainView` (базовая ветка копии);
3. копия и **оригинал** `llada_diffusion_generate` на `PlainView` (базовая ветка оригинала);
4. копия и **оригинал** на `CoddLlada` с `use_pc=False` (путь `CoddLlada`).

Кроме токенов проверяются: структура журнала шагов (число раскрытых позиций на шаге, позиции внутри текущего блока, нет повторного раскрытия, токены в журнале = токены в результате, уверенность конечна и убывает внутри шага), промпт (содержит system и вопрос) и путь оценки на синтетических ответах (эталон из датасета должен получить `correct=1`, искажённый - `0`). Дополнительно проверяется, что логиты конечны (защита от NaN/inf, особенно при `DTYPE="float16"`). При **любом** расхождении выбрасывается ошибка с описанием первой несостыковки, и ячейка полного прогона не запустится (`SMOKE_OK` остаётся `False`). В конце (ячейка 5.2) измеряется **пиковая видеопамять** при реальном `GEN_LENGTH` на самой длинной по токенам задаче (`assert`: запас не менее 2 ГиБ от общего объёма), печатаются секунды на задачу и прогноз времени на 4 комбинации x 100 задач. Между задачами вызывается `torch.cuda.empty_cache()`.

In [ ]:
# ===================== 5.1 ДЫМОВОЙ ТЕСТ: копия == оригинал == путь CoddLlada(use_pc=False) =====================
import io
import contextlib

SMOKE_N_TASKS = 3
SMOKE_STEPS = 8
SMOKE_GEN_LENGTH = 64
SMOKE_BLOCK_LENGTH = 32
SMOKE_OK = False
assert SMOKE_GEN_LENGTH % SMOKE_BLOCK_LENGTH == 0 and SMOKE_STEPS % (SMOKE_GEN_LENGTH // SMOKE_BLOCK_LENGTH) == 0
smoke_insts = INSTANCES[:SMOKE_N_TASKS]
assert len(smoke_insts) == SMOKE_N_TASKS

# ---------- A. путь оценки на синтетических ответах (без GPU) ----------
print("A. Проверка пути оценки на синтетических ответах")
for inst in smoke_insts:
    gold_text = inst.doc["answer"]
    s_ok, p_ok = score_response(inst, gold_text)
    assert s_ok["flexible-extract"] == 1.0 and s_ok["strict-match"] == 1.0, (
        f"эталонный ответ задачи {inst.doc_id} оценён как неверный: предсказание {p_ok}, эталон {gold_final(gold_text)!r} - путь оценки сломан")
    gold_num = gold_final(gold_text).replace(",", "")
    try:
        wrong_num = str(int(float(gold_num)) + 1)
    except ValueError:
        print(f"   задача {inst.doc_id}: эталон не число ({gold_num!r}), проверка искажённого ответа пропущена")
        continue
    wrong_text = gold_text.rsplit("####", 1)[0] + "#### " + wrong_num
    s_bad, p_bad = score_response(inst, wrong_text)
    assert s_bad["flexible-extract"] == 0.0 and s_bad["strict-match"] == 0.0, (
        f"искажённый ответ задачи {inst.doc_id} оценён как верный: {p_bad} - путь оценки сломан")
    print(f"   задача {inst.doc_id}: эталон -> верно ({p_ok['flexible-extract']!r}); искажённый -> неверно ({p_bad['flexible-extract']!r})")

# ---------- B. промпт ----------
print("B. Проверка промпта")
for inst in smoke_insts:
    ctx_text = inst.args[0]
    assert SYSTEM_INSTRUCTION in ctx_text, f"в промпте задачи {inst.doc_id} нет system-инструкции"
    assert inst.doc["question"] in ctx_text, f"в промпте задачи {inst.doc_id} нет текста вопроса"
    manual = lm.apply_chat_template(
        [{"role": "system", "content": SYSTEM_INSTRUCTION}, {"role": "user", "content": task.doc_to_text(inst.doc)}],
        add_generation_prompt=True)
    print(f"   задача {inst.doc_id}: совпадает с ручной сборкой system+user через chat template: {manual == ctx_text}")


# ---------- C. токены: копия == оригинал на GPU ----------
def quiet_call(fn, *args, **kwargs):
    # оригинальная функция печатает tqdm в stderr; на результат это не влияет
    with contextlib.redirect_stderr(io.StringIO()):
        return fn(*args, **kwargs)


def assert_same(name_a, a, name_b, b, prompt_len, where, hint=""):
    if a.shape == b.shape and bool((a == b).all()):
        return
    if a.shape == b.shape:
        diff_pos = (a != b).nonzero()[:, 1].tolist()
        first = diff_pos[0]
        detail = (f"позиций отличается: {len(diff_pos)}; первая: {first} (позиция {first - prompt_len} в ответе); "
                  f"токены {a[0, first].item()} ({name_a}) и {b[0, first].item()} ({name_b})")
    else:
        detail = f"формы различаются: {tuple(a.shape)} и {tuple(b.shape)}"
    raise AssertionError(f"РАСХОЖДЕНИЕ [{where}]: {name_a} != {name_b}. {detail}. {hint}"
                         "Остановка: полный прогон запрещён, пока причина не найдена.")


def check_steplog(steplog, x, prompt_len, gen_length, block_length, num_steps, where):
    num_blocks = gen_length // block_length
    steps = num_steps // num_blocks
    assert len(steplog) == num_steps, f"[{where}] в журнале {len(steplog)} шагов вместо {num_steps}"
    seen = set()
    revealed_in_block = {}
    for k, rec in enumerate(steplog):
        b, i = divmod(k, steps)
        assert (rec["block"], rec["step_in_block"], rec["global_step"]) == (b, i, k), f"[{where}] нарушен порядок шагов на {k}"
        start, end = prompt_len + b * block_length, prompt_len + (b + 1) * block_length
        expected_n = block_length // steps + (1 if i < block_length % steps else 0)
        pos = rec["positions_abs"]
        assert len(pos) == expected_n, f"[{where}] шаг {k}: раскрыто {len(pos)} позиций вместо {expected_n}"
        assert all(start <= p < end for p in pos), f"[{where}] шаг {k}: позиции вне текущего блока [{start}, {end}): {pos}"
        assert not (set(pos) & seen), f"[{where}] шаг {k}: позиция раскрыта повторно"
        seen |= set(pos)
        assert rec["positions_rel"] == [p - prompt_len for p in pos], f"[{where}] шаг {k}: positions_rel не согласованы"
        assert rec["token_ids"] == [x[0, p].item() for p in pos], f"[{where}] шаг {k}: токены журнала не совпали с результатом"
        done_before = revealed_in_block.get(b, 0)
        assert rec["n_masked_block_before"] == block_length - done_before, f"[{where}] шаг {k}: неверное число масок в блоке"
        revealed_in_block[b] = done_before + len(pos)
        conf = rec["confidence"]
        assert all(math.isfinite(c) and -1e-2 <= c <= 1 + 1e-2 for c in conf), f"[{where}] шаг {k}: уверенность вне [0, 1] или не конечна: {conf}"
        assert all(conf[t] >= conf[t + 1] for t in range(len(conf) - 1)), f"[{where}] шаг {k}: уверенность не убывает внутри шага"
    assert seen == set(range(prompt_len, prompt_len + gen_length)), f"[{where}] раскрыты не все позиции ответа"


print("C0. Логиты конечны (защита от NaN/inf; особенно важно при DTYPE=float16)")
for inst in smoke_insts:
    _P = len(PROMPT_IDS[inst.doc_id])
    _probe = torch.full((1, _P + SMOKE_GEN_LENGTH), MASK_ID, dtype=torch.long, device=model.device)
    _probe[0, :_P] = torch.tensor(PROMPT_IDS[inst.doc_id], dtype=torch.long, device=model.device)
    with torch.no_grad():
        _logits = model(_probe).logits
    assert bool(torch.isfinite(_logits).all()), (
        f"В логитах задачи {inst.doc_id} есть NaN/inf при DTYPE={DTYPE!r}. Остановка: этот dtype непригоден (решение за автором проекта).")
    print(f"   задача {inst.doc_id}: логиты конечны, форма {tuple(_logits.shape)}, dtype {_logits.dtype}")
    del _probe, _logits
    torch.cuda.empty_cache()

print("C. Токены: копия против оригинала (GPU)")
smoke_samples = {}
for remasking in REMASKING_LIST:
    for inst in smoke_insts:
        prompt = torch.tensor([PROMPT_IDS[inst.doc_id]], dtype=torch.long, device=model.device)
        P = prompt.shape[1]
        common = dict(num_steps=SMOKE_STEPS, gen_length=SMOKE_GEN_LENGTH, block_length=SMOKE_BLOCK_LENGTH,
                      temperature=TEMPERATURE, cfg_scale=0., remasking=remasking, mask_id=MASK_ID)
        where = f"remasking={remasking}, задача {inst.doc_id}"

        x_copy, log_copy = llada_generate_logged(model, prompt, use_pc=False, **common)
        x_copy_2, _ = llada_generate_logged(model, prompt, use_pc=False, **common)
        x_copy_plain, _ = llada_generate_logged(PLAIN, prompt, use_pc=False, **common)
        x_orig_plain = quiet_call(llada_diffusion_generate, PLAIN, prompt, **common)
        x_orig_codd = quiet_call(llada_diffusion_generate, model, prompt, use_pc=False, **common)

        # 1) детерминизм: если не совпало, виноват GPU, а не логика копии
        assert_same("копия (прогон 2)", x_copy_2, "копия (прогон 1)", x_copy, P, where,
                    hint="Копия расходится сама с собой: недетерминизм GPU/ядер, а не ошибка логики. ")
        # 2) обе ветки копии
        assert_same("копия на PlainView", x_copy_plain, "копия на CoddLlada", x_copy, P, where)
        # 3) копия против оригинала, базовая ветка
        assert_same("копия на PlainView", x_copy_plain, "оригинал на PlainView", x_orig_plain, P, where)
        # 4) копия против оригинала, путь CoddLlada(use_pc=False)
        assert_same("копия на CoddLlada", x_copy, "оригинал на CoddLlada(use_pc=False)", x_orig_codd, P, where)

        assert bool((x_copy[:, :P] == prompt).all()), f"[{where}] промпт в результате изменился"
        assert not bool((x_copy[:, P:] == MASK_ID).any()), f"[{where}] в ответе остались маски"
        check_steplog(log_copy, x_copy, P, SMOKE_GEN_LENGTH, SMOKE_BLOCK_LENGTH, SMOKE_STEPS, where)
        smoke_samples[(remasking, inst.doc_id)] = tokenizer.decode(x_copy[0, P:].tolist(), skip_special_tokens=True)
        print(f"   OK [{where}]: 5 прогонов совпали по токенам, журнал согласован")
        torch.cuda.empty_cache()   # между задачами

print()
print("Пример ответа дымового теста (задача 0, low_confidence, 8 шагов, 64 токена; текст может быть бессмысленным - шагов мало):")
print(repr(smoke_samples[(REMASKING_LIST[0], 0)][:300]))
print("ДЫМОВОЙ ТЕСТ (корректность): ПРОЙДЕН")

In [ ]:
# ===================== 5.2 СЕКУНДЫ НА ЗАДАЧУ И ПРОГНОЗ ВРЕМЕНИ =====================
# --- пиковая видеопамять при РЕАЛЬНОМ GEN_LENGTH на самой длинной по токенам задаче ---
# Форма тензоров не зависит от числа шагов, поэтому берём минимально допустимое число шагов (по одному на блок).
# low_confidence считает softmax в float64 и самый прожорливый, поэтому проверяются все remasking из списка.
GIB = 1024 ** 3
longest_inst = max(INSTANCES, key=lambda i: len(PROMPT_IDS[i.doc_id]))
total_mem = torch.cuda.get_device_properties(GPU_INDEX).total_memory
PEAK_REPORT = {}
for _rm in REMASKING_LIST:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    run_generation(longest_inst, _rm, NUM_BLOCKS)
    _peak_alloc = torch.cuda.max_memory_allocated()
    _peak_res = torch.cuda.max_memory_reserved()
    PEAK_REPORT[_rm] = {"peak_allocated_gib": _peak_alloc / GIB, "peak_reserved_gib": _peak_res / GIB,
                        "headroom_gib": (total_mem - _peak_alloc) / GIB}
    print(f"Пик памяти [{_rm}], самая длинная задача ({len(PROMPT_IDS[longest_inst.doc_id])} ток. промпта + {GEN_LENGTH} ответа): "
          f"allocated {_peak_alloc / GIB:.2f} ГиБ, reserved {_peak_res / GIB:.2f} ГиБ из {total_mem / GIB:.2f} ГиБ; запас (total - allocated) {(total_mem - _peak_alloc) / GIB:.2f} ГиБ")
    assert total_mem - _peak_alloc >= 2 * GIB, (
        f"Запас видеопамяти {(total_mem - _peak_alloc) / GIB:.2f} ГиБ < 2 ГиБ при remasking={_rm}, GEN_LENGTH={GEN_LENGTH}, DTYPE={DTYPE}. "
        "Полный прогон может упасть по OOM. Остановка: решение (меньше GEN_LENGTH, другой dtype и т.п.) за автором проекта.")
    if total_mem - _peak_res < 2 * GIB:
        print(f"   ВНИМАНИЕ: с учётом кэша аллокатора (reserved) запас {(total_mem - _peak_res) / GIB:.2f} ГиБ < 2 ГиБ; empty_cache между задачами включён.")
    torch.cuda.empty_cache()

TIMING_N_TASKS = 2
timing_remasking = REMASKING_LIST[0]
timing_steps = NUM_STEPS_LIST[0]
timing_secs = []
for inst in INSTANCES[:TIMING_N_TASKS]:
    _x, _log, _plen, _sec = run_generation(inst, timing_remasking, timing_steps)   # реальные GEN_LENGTH и BLOCK_LENGTH
    timing_secs.append(_sec)
    torch.cuda.empty_cache()   # между задачами
sec_per_task = sum(timing_secs) / len(timing_secs)
sec_per_forward = sec_per_task / timing_steps
print(f"Замер: {TIMING_N_TASKS} задачи, remasking={timing_remasking}, num_steps={timing_steps}, GEN_LENGTH={GEN_LENGTH}")
print(f"Секунд на задачу: {sec_per_task:.2f} ({timing_steps} шагов), на один проход модели: {sec_per_forward:.3f} c")

n_eff = RUN_LIMIT if RUN_LIMIT is not None else N_TASKS
per_combo = {}
print()
print(f"Прогноз времени (время считается пропорциональным числу проходов модели; оценка по замеру выше, НЕ гарантия; задач на комбинацию: {n_eff}):")
for _rm in REMASKING_LIST:
    for _st in NUM_STEPS_LIST:
        per_combo[(_rm, _st)] = sec_per_forward * _st * n_eff
        print(f"   {_rm:<16} steps={_st:<4} ~{sec_per_forward * _st:6.1f} c/задачу, ~{datetime.timedelta(seconds=int(per_combo[(_rm, _st)]))} на комбинацию")
total_forecast = sum(per_combo.values())
print(f"ИТОГО на {len(per_combo)} комбинации x {n_eff} задач: ~{datetime.timedelta(seconds=int(total_forecast))} (~{total_forecast / 3600:.1f} ч)")
print("Секунд на задачу по числу шагов:", {st: round(sec_per_forward * st, 1) for st in NUM_STEPS_LIST})
if MAX_RUNTIME_MIN is not None:
    print(f"MAX_RUNTIME_MIN={MAX_RUNTIME_MIN}: за одну сессию будет сделано примерно {MAX_RUNTIME_MIN * 60 / total_forecast * 100:.0f}% полного прогона (оценка).")
print("(лимиты сессии DataSphere на длительность неизвестны: НЕ ПРОВЕРЕНО; благодаря кэшу после каждой задачи прогон можно продолжить)")

SMOKE_OK = True
write_json_atomic(os.path.join(OUT_DIR, "smoke.json"), {
    "ok": True, "finished_utc": utc_now(), "exp_sha256": EXP_SHA256, "gen_length": GEN_LENGTH,
    "sec_per_task_first_steps": sec_per_task, "sec_per_forward": sec_per_forward,
    "forecast_seconds_total": total_forecast, "forecast_tasks_per_combo": n_eff,
    "pyjuice_mode": PYJUICE_MODE, "dtype": DTYPE, "gpu_name": GPU_NAME, "gpu_capability": GPU_CAPABILITY,
    "peak_memory": PEAK_REPORT, "total_gpu_memory_gib": total_mem / GIB,
})
print()
print("SMOKE_OK = True: полный прогон (раздел 6) разрешён.")

## 6. Полный прогон

4 комбинации (`remasking` x `num_steps`) по `N_TASKS` задач, `use_pc=False`, `temperature=0` (жадный выбор), один запуск на комбинацию (`seed0`). Прогон **идемпотентен**: после обрыва сессии DataSphere просто перезапустите ячейки по порядку (конфиг, окружение, модель, задачи, функции, дымовой тест) и затем эту ячейку - готовые задачи будут пропущены. Запуск возможен только после успешного дымового теста (`SMOKE_OK`).

`RUN_LIMIT` (раздел 0) ограничивает число обрабатываемых задач; при `RUN_LIMIT != None` прогон помечается как **неполный**.

`MAX_RUNTIME_MIN` (раздел 0, по умолчанию `None`): если задан, то после первой завершённой задачи, на которой истёк лимит времени (отсчёт от начала этой ячейки), прогон корректно останавливается: кэш и steplog к этому моменту уже дописаны на диск. Остановка происходит только между задачами (одна задача при 128 шагах может идти минуты). Ячейка напечатает, как продолжить. **Машину блокнот остановить не может**: по окончании остановите JupyterLab и ВМ в интерфейсе DataSphere.

In [ ]:
# ===================== 6.1 ПОЛНЫЙ ПРОГОН =====================
assert globals().get("SMOKE_OK") is True, "Сначала успешно выполните дымовой тест (раздел 5): SMOKE_OK не равен True."
assert model.pc_model is None and USE_PC is False

run_instances = INSTANCES if RUN_LIMIT is None else INSTANCES[:RUN_LIMIT]
if RUN_LIMIT is not None:
    print("=" * 90)
    print(f"ВНИМАНИЕ: RUN_LIMIT={RUN_LIMIT}. Будут обработаны только первые {RUN_LIMIT} задач из {N_TASKS}. ПРОГОН НЕПОЛНЫЙ.")
    print("=" * 90)
print(f"GEN_LENGTH={GEN_LENGTH}, BLOCK_LENGTH={BLOCK_LENGTH}, задач: {len(run_instances)}, комбинаций: {len(REMASKING_LIST) * len(NUM_STEPS_LIST)}")

_t_all = time.time()
_deadline = None if MAX_RUNTIME_MIN is None else _t_all + 60 * MAX_RUNTIME_MIN
if _deadline is not None:
    print(f"MAX_RUNTIME_MIN={MAX_RUNTIME_MIN}: после завершённой задачи, на которой истечёт лимит, прогон корректно остановится (кэш и steplog будут дописаны).")
STOPPED_BY_TIMER = False
for remasking in REMASKING_LIST:
    for num_steps in NUM_STEPS_LIST:
        if _deadline is not None and time.time() >= _deadline:
            STOPPED_BY_TIMER = True
            break
        if run_combo(remasking, num_steps, run_instances, deadline=_deadline) == "stopped":
            STOPPED_BY_TIMER = True
            break
    if STOPPED_BY_TIMER:
        break
print(f"Цикл прогона завершён за {datetime.timedelta(seconds=int(time.time() - _t_all))} (в этой сессии), остановка по таймеру: {STOPPED_BY_TIMER}.")

# ---------- итоговая проверка файлов ----------
for remasking in REMASKING_LIST:
    for num_steps in NUM_STEPS_LIST:
        present = sorted(r["task_id"] for r in read_jsonl_tolerant(result_path(remasking, num_steps)))
        if STOPPED_BY_TIMER:
            validate_combo(remasking, num_steps, present)   # прогон неполный: проверяем только согласованность готового
        elif RUN_LIMIT is None:
            validate_combo(remasking, num_steps, TASK_IDS)
        else:
            wanted = [inst.doc_id for inst in run_instances]
            assert set(wanted) <= set(present), f"{remasking}/{num_steps}: нет результатов для задач {sorted(set(wanted) - set(present))}"
            validate_combo(remasking, num_steps, present)
        print(f"   проверено: {os.path.basename(result_path(remasking, num_steps))}: {len(present)} задач, steplog согласован")

if STOPPED_BY_TIMER:
    print("=" * 100)
    print(f"ПРОГОН ОСТАНОВЛЕН по MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} (НЕ завершён). Кэш и steplog дописаны, файлы согласованы.")
    print("Как продолжить:")
    print("   - ядро живо: выполните эту же ячейку (6.1) ещё раз, готовые задачи будут пропущены;")
    print("   - ядро перезапущено или сессия оборвалась: выполните по порядку ячейки конфига, раздела 1, моделей, задач, функций, дымовой тест (разделы 0-5) и затем 6.1.")
    print("   Для снятия ограничения поставьте MAX_RUNTIME_MIN = None.")
    print("НАПОМИНАНИЕ: блокнот не может остановить машину. Остановите JupyterLab и ВМ в интерфейсе DataSphere (иначе продолжится тарификация).")
    print("=" * 100)
elif RUN_LIMIT is None:
    print(f"ГОТОВО: все {len(REMASKING_LIST) * len(NUM_STEPS_LIST)} комбинации по {N_TASKS} задач, файлы в {OUT_DIR}")
else:
    print(f"НЕПОЛНЫЙ ПРОГОН (RUN_LIMIT={RUN_LIMIT}). Для полного прогона верните RUN_LIMIT = None и перезапустите эту ячейку.")

## 7. Сводка

Точность (доля верных ответов под фильтром `flexible-extract`, как в `eval/utils.py::parse_results`) с 95% доверительным интервалом (percentile bootstrap по задачам: задачи ресэмплируются с возвратом, `N_BOOTSTRAP` повторов, фиксированный seed только для bootstrap) для каждой комбинации `remasking` x число шагов. Дополнительно - `strict-match`. Сравнений с другими экспериментами здесь нет: их результатов этот блокнот не видит.

In [ ]:
# ===================== 7.1 ТАБЛИЦА ТОЧНОСТИ С BOOTSTRAP-ИНТЕРВАЛАМИ =====================
summary_rows = []
for remasking in REMASKING_LIST:
    for num_steps in NUM_STEPS_LIST:
        rows = sorted(read_jsonl_tolerant(result_path(remasking, num_steps)), key=lambda r: r["task_id"])
        if not rows:
            print(f"нет результатов для {remasking}/{num_steps}")
            continue
        acc, lo, hi = bootstrap_ci([r["correct"] for r in rows])
        acc_s, lo_s, hi_s = bootstrap_ci([r["correct_strict"] for r in rows])
        summary_rows.append({
            "remasking": remasking, "num_steps": num_steps, "n": len(rows),
            "accuracy": acc, "ci_low": lo, "ci_high": hi,
            "accuracy_strict": acc_s, "ci_low_strict": lo_s, "ci_high_strict": hi_s,
        })

print(f"exp1 (обычная LLaDA, HMM не используется) | GSM8K test, первые {N_TASKS} задач | GEN_LENGTH={GEN_LENGTH}, BLOCK_LENGTH={BLOCK_LENGTH}, greedy | bootstrap: {N_BOOTSTRAP} повторов, 95%")
print(f"sha256 списка id задач: {IDS_SHA256}")
print()
header = f"{'remasking':<16}{'steps':>6}{'n':>5}   {'accuracy (flexible-extract)':<30}{'accuracy (strict-match)':<28}"
print(header)
print("-" * len(header))
for s in summary_rows:
    flag = "" if s["n"] == N_TASKS else "  <- НЕПОЛНО"
    print(f"{s['remasking']:<16}{s['num_steps']:>6}{s['n']:>5}   "
          f"{100 * s['accuracy']:5.1f}% [{100 * s['ci_low']:5.1f}; {100 * s['ci_high']:5.1f}]".ljust(60)
          + f"{100 * s['accuracy_strict']:5.1f}% [{100 * s['ci_low_strict']:5.1f}; {100 * s['ci_high_strict']:5.1f}]{flag}")
print()
print("Доверительный интервал отражает только выбор 100 задач (разброс по задачам), а не случайность генерации: случайности здесь нет.")

SUMMARY_COMPLETE = (len(summary_rows) == len(REMASKING_LIST) * len(NUM_STEPS_LIST)) and all(s["n"] == N_TASKS for s in summary_rows)
write_json_atomic(os.path.join(OUT_DIR, "summary.json"), {
    "experiment": EXP_NAME, "gen_length": GEN_LENGTH, "block_length": BLOCK_LENGTH, "n_tasks": N_TASKS,
    "ids_sha256": IDS_SHA256, "n_bootstrap": N_BOOTSTRAP, "bootstrap_seed": BOOTSTRAP_SEED,
    "complete": SUMMARY_COMPLETE, "rows": summary_rows,
})
print("summary.json записан в", OUT_DIR, "| полный:", SUMMARY_COMPLETE)

In [ ]:
# ===================== 7.2 (НЕОБЯЗАТЕЛЬНО) ЦИФРЫ ИЗ СТАТЬИ - ТОЛЬКО РУЧНОЙ ВВОД =====================
# Блокнот ничего не подставляет сам. Если нужно, впишите цифры из таблицы статьи вручную, в процентах.
# Формат ключа: (remasking, num_steps) -> число, например: {("low_confidence", 64): <число из статьи>}
PAPER_NUMBERS = {}

if not PAPER_NUMBERS:
    print("PAPER_NUMBERS пуст - вывод пропущен (так и задумано).")
else:
    print("ВНИМАНИЕ: цифры статьи получены в других условиях (в статье длина генерации 512 токенов и полный test GSM8K, а не первые "
          f"{N_TASKS} задач; здесь GEN_LENGTH={GEN_LENGTH}). Прямое сравнение корректно только после согласования условий.")
    print("Также НЕ ПРОВЕРЕНО, на каком коде посчитан margin в статье (оригинальный харнесс без HMM поддерживает только low_confidence и random).")
    ours = {(s["remasking"], s["num_steps"]): 100 * s["accuracy"] for s in summary_rows}
    print(f"{'remasking':<16}{'steps':>6}{'наш результат, %':>20}{'из статьи, % (ввод вручную)':>32}")
    for key, paper_value in PAPER_NUMBERS.items():
        mine = ours.get(key)
        print(f"{key[0]:<16}{key[1]:>6}{('нет' if mine is None else f'{mine:.1f}'):>20}{paper_value:>32}")

In [ ]:
# ===================== 7.3 ФАКТЫ ДЛЯ ОТЧЁТА =====================
print("pyjuice:", PYJUICE_MODE, "(в блокнотах 2-5 заглушка ЗАПРЕЩЕНА)")
print("HMM_CKPT:", HMM_CKPT, "(изменён по решению автора; в exp1 не используется)")
print("GEN_LENGTH:", GEN_LENGTH, "| DTYPE:", DTYPE, "| GPU:", GPU_NAME, "capability", GPU_CAPABILITY)
print("Сверка кода модели (sha256):", {k: v["verdict"] for k, v in MODEL_CODE_CHECK.items()})
print("sha256 списка id задач:", IDS_SHA256)
print("sha256 общих параметров:", COMMON_SHA256)
print("Результаты:", OUT_DIR)
for _name in sorted(os.listdir(OUT_DIR)):
    print("   ", _name)
print()
print("!" * 100)
print("НАПОМИНАНИЕ: этот блокнот НЕ может остановить машину. После окончания работы вручную остановите JupyterLab и ВМ")
print("(виртуальную машину/ресурс проекта) в интерфейсе Yandex DataSphere, иначе продолжится тарификация GPU.")
print("Точные пункты меню НЕ ПРОВЕРЕНЫ - ищите в интерфейсе проекта остановку ВМ/ноутбука.")
print("!" * 100)